# COST — External Validation on eICU-CRD

This notebook is the FULL-data version of the validated DEMO pipeline. It uses `DATASET_MODE = 'FULL'`, retains the finalized feature mappings and COST Improved inference pipeline, and is intended for external validation without retraining/fine-tuning.

**Note:** FULL eICU-CRD requires authorized PhysioNet access. All previous DEMO outputs have been cleared to avoid mixing DEMO and FULL results.


# COST (Improved) — External Validation on eICU-CRD

**Purpose:** external multicenter validation of the already-trained COST (Improved) model.  
**Important:** this notebook **does not retrain COST**. It harmonizes eICU variables to the original 48 h × 31 temporal + 11 static representation, applies the **MIMIC-IV training normalization stored in `dsrnn_v2_best.pt`**, loads the original checkpoint, and evaluates eICU.

Two modes are provided:
- `DEMO`: automatically downloads the open eICU demo and verifies the full pipeline/schema.
- `FULL`: downloads the credentialed full eICU-CRD v2.0 after you enter your PhysioNet username/password. **Use FULL results in the paper.**

The notebook deliberately stops/report flags when feature compatibility is poor rather than silently changing the COST architecture.

In [ ]:
# CELL 1 — Colab / Drive / packages
from google.colab import drive
drive.mount('/content/drive')

!pip -q install pandas numpy scikit-learn torch tqdm requests

import os, re, math, json, time, getpass, warnings, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

In [ ]:
# CELL 2 — CONFIGURATION
# Start with DEMO to verify schema/pipeline. For paper results change to FULL.
DATASET_MODE = 'FULL'   # 'DEMO' or 'FULL'

ROOT = Path('/content/drive/MyDrive/COST_PROJECT')
MIMIC_BASE = ROOT / 'mimic-iv-2.1'
EICU_DIR = ROOT / ('eicu-crd-demo-2.0.1' if DATASET_MODE == 'DEMO' else 'eicu-crd-2.0')
OUT_DIR = MIMIC_BASE / 'revision2_outputs' / 'external_eicu'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Original COST Improved checkpoint saved by COST_Master_AllExperiments_CLEAN.ipynb
CKPT_CANDIDATES = [
    MIMIC_BASE / 'saved_models' / 'dsrnn_v2_best.pt',
    ROOT / 'saved_models' / 'dsrnn_v2_best.pt',
]
CKPT_PATH = next((p for p in CKPT_CANDIDATES if p.exists()), CKPT_CANDIDATES[0])

print('Mode       :', DATASET_MODE)
print('eICU dir   :', EICU_DIR)
print('Output dir :', OUT_DIR)
print('Checkpoint :', CKPT_PATH, 'FOUND' if CKPT_PATH.exists() else 'NOT FOUND')

In [ ]:
# CELL 4 — ORIGINAL COST FEATURE CONTRACT (exact order)
VITAL_SIGNS = ['heart_rate','sbp','dbp','map','resp_rate','spo2','temperature']
LAB_VALUES = [
    'creatinine','bnp','lactate','troponin','sodium','potassium','chloride','bicarbonate',
    'bun','glucose','hemoglobin','wbc','platelets','inr','pt','ptt','ph','pao2','paco2','fio2',
    'albumin','bilirubin_total','alt','ast'
]
TEMP_FEATURES = VITAL_SIGNS + LAB_VALUES
STATIC_FEATURES = [
    'age','gender_male','admission_emergency','admission_urgent','elixhauser_score',
    'hypertension','diabetes','ckd','copd','af','prior_hf'
]
assert len(TEMP_FEATURES)==31 and len(STATIC_FEATURES)==11
N_FEATURES=31; N_STATIC=11; N_INPUT=93
print('Temporal:', len(TEMP_FEATURES), TEMP_FEATURES)
print('Static  :', len(STATIC_FEATURES), STATIC_FEATURES)

In [ ]:
from pathlib import Path

EICU_DIR = Path(
    "/content/drive/MyDrive/COST_PROJECT/"
    "mimic-iv-2.1/revision2_outputs/"
    "external_eicu/eicu_full"
)

print("EICU_DIR:", EICU_DIR)
print("Exists:", EICU_DIR.exists())

for f in sorted(EICU_DIR.glob("*.csv.gz")):
    print("✓", f.name)

In [ ]:
# CELL 5 — LOAD eICU CORE TABLES + schema audit
def read_gz(name, usecols=None):
    p=EICU_DIR/name
    return pd.read_csv(p, compression='gzip', low_memory=False, usecols=usecols)

patient = read_gz('patient.csv.gz')
diagnosis = read_gz('diagnosis.csv.gz')
admissiondx = read_gz('admissionDx.csv.gz')
pasthistory = read_gz('pastHistory.csv.gz')

print('patient:', patient.shape)
print('diagnosis:', diagnosis.shape)
print('admissionDx:', admissiondx.shape)
print('pastHistory:', pasthistory.shape)
print('patient columns: ', patient.columns.tolist())
print('diagnosis columns: ', diagnosis.columns.tolist())
print('pastHistory columns: ', pasthistory.columns.tolist())

In [ ]:
# CELL 6 — eICU HEART-FAILURE COHORT
# HF is identified from structured diagnosis/admission diagnosis text.
def textcol(df, candidates):
    for c in candidates:
        if c in df.columns: return c
    return None

dx_col = textcol(diagnosis, ['diagnosisstring','diagnosisString'])
adx_col = textcol(admissiondx, ['admitdxpath','admitDxPath','admitdxname'])

hf_regex = r'heart failure|congestive heart failure|CHF|cardiomyopathy.*heart failure'
hf_ids=set()
if dx_col:
    hf_ids |= set(diagnosis.loc[diagnosis[dx_col].astype(str).str.contains(hf_regex, case=False, regex=True, na=False), 'patientunitstayid'])
if adx_col:
    hf_ids |= set(admissiondx.loc[admissiondx[adx_col].astype(str).str.contains(hf_regex, case=False, regex=True, na=False), 'patientunitstayid'])

cohort = patient[patient.patientunitstayid.isin(hf_ids)].copy()
# Adult
age_num = pd.to_numeric(cohort['age'].astype(str).str.replace('> 89','90',regex=False), errors='coerce')
cohort['age_num']=age_num
cohort=cohort[cohort.age_num>=18].copy()
# ICU LOS >24h: unitdischargeoffset is minutes from unit admission
if 'unitdischargeoffset' in cohort.columns:
    cohort=cohort[pd.to_numeric(cohort.unitdischargeoffset,errors='coerce')>24*60].copy()
# outcome
cohort['mortality'] = cohort['hospitaldischargestatus'].astype(str).str.lower().eq('expired').astype(int)

# one ICU stay per unique patient-hospital stay: earliest unit visit
sort_cols=[c for c in ['patienthealthsystemstayid','unitvisitnumber','patientunitstayid'] if c in cohort.columns]
if 'patienthealthsystemstayid' in cohort.columns:
    cohort=cohort.sort_values(sort_cols).drop_duplicates('patienthealthsystemstayid',keep='first')
else:
    cohort=cohort.sort_values('patientunitstayid').drop_duplicates('uniquepid',keep='first')

print('HF external cohort before minimum-data filter:', len(cohort))
print('Mortality:', cohort.mortality.sum(), f'({cohort.mortality.mean()*100:.2f}%)')
print('Hospitals:', cohort['hospitalid'].nunique() if 'hospitalid' in cohort else 'not available in demo')

In [ ]:
# CELL 7 — TEMPORAL MAPPING: vitals, 0–48h hourly
ids=set(cohort.patientunitstayid.astype(int))
vp = read_gz('vitalPeriodic.csv.gz')
vp = vp[vp.patientunitstayid.isin(ids)].copy()
vp = vp[(pd.to_numeric(vp.observationoffset,errors='coerce')>=0) & (pd.to_numeric(vp.observationoffset,errors='coerce')<48*60)]
vp['hour']=(pd.to_numeric(vp.observationoffset,errors='coerce')//60).astype(int)

vmap={
 'heart_rate':['heartrate'], 'sbp':['systemicsystolic'], 'dbp':['systemicdiastolic'],
 'map':['systemicmean'], 'resp_rate':['respiration'], 'spo2':['sao2'], 'temperature':['temperature']
}
for dst, srcs in vmap.items():
    src=next((s for s in srcs if s in vp.columns),None)
    vp[dst]=pd.to_numeric(vp[src],errors='coerce') if src else np.nan

vital_hour=vp.groupby(['patientunitstayid','hour'])[VITAL_SIGNS].median().reset_index()
print('Vital hourly rows:',len(vital_hour))
print(vital_hour[VITAL_SIGNS].notna().mean().sort_values())

In [ ]:
# ============================================================
# CELL 7B — VITAL MAPPING AUDIT
# Current external validation uses periodic vitals only
# ============================================================

print("=" * 85)
print("FINAL eICU VITAL TABLE — PERIODIC VITALS")
print("=" * 85)

print(f"Rows     : {len(vital_hour):,}")
print(
    f"Patients : "
    f"{vital_hour['patientunitstayid'].nunique():,}"
)

print("\nFeature availability:")

for f in VITAL_SIGNS:
    n = vital_hour[f].notna().sum()
    pct = vital_hour[f].notna().mean() * 100

    print(
        f"{f:15s}: "
        f"{n:8,d} ({pct:6.2f}%)"
    )

print("\nAperiodic BP : NOT USED")
print("Reason       : vitalAperiodic.csv.gz incomplete/corrupt")
print("0–48 h       : YES")
print("Training     : NONE")

print("=" * 85)
print("✓ VITAL MAPPING COMPLETE")
print("=" * 85)

In [ ]:
# ============================================================
# CELL 8 — eICU LAB MAPPING, 0–48 HOURS
# NO TRAINING
# ============================================================

import pandas as pd
import numpy as np
import gc

print("=" * 85)
print("eICU LAB MAPPING — 0–48 HOURS")
print("=" * 85)

# ------------------------------------------------------------
# 1. LOAD LAB
# ------------------------------------------------------------

lab_path = EICU_DIR / "lab.csv.gz"

print("Lab file:", lab_path)
print("Exists  :", lab_path.exists())

if not lab_path.exists():
    raise FileNotFoundError("lab.csv.gz not found")

print(
    f"Size    : "
    f"{lab_path.stat().st_size / 1024**2:.1f} MB"
)

print("\nReading lab.csv.gz ...")

try:
    lab = pd.read_csv(
        lab_path,
        compression="gzip",
        low_memory=False
    )
except EOFError as e:
    raise RuntimeError(
        "lab.csv.gz is incomplete/corrupt."
    ) from e

print(f"✓ Raw lab rows: {len(lab):,}")


# ------------------------------------------------------------
# 2. FILTER TO OUR HF COHORT
# ------------------------------------------------------------

lab["patientunitstayid"] = pd.to_numeric(
    lab["patientunitstayid"],
    errors="coerce"
)

lab = lab[
    lab["patientunitstayid"].isin(ids)
].copy()

print(
    f"✓ HF cohort lab rows: "
    f"{len(lab):,}"
)


# ------------------------------------------------------------
# 3. FILTER 0–48 HOURS
# ------------------------------------------------------------

lab["labresultoffset"] = pd.to_numeric(
    lab["labresultoffset"],
    errors="coerce"
)

lab = lab[
    (lab["labresultoffset"] >= 0) &
    (lab["labresultoffset"] < 48 * 60)
].copy()

lab["hour"] = (
    lab["labresultoffset"] // 60
).astype(int)

lab["labresult"] = pd.to_numeric(
    lab["labresult"],
    errors="coerce"
)

lab["labname_clean"] = (
    lab["labname"]
    .astype(str)
    .str.lower()
    .str.strip()
)

print(
    f"✓ 0–48 h lab rows: "
    f"{len(lab):,}"
)


# ------------------------------------------------------------
# 4. MAP eICU LAB NAMES -> COST FEATURES
# ------------------------------------------------------------

LAB_ALIASES = {

    "creatinine": [
        "creatinine"
    ],

    "bnp": [
        "bnp",
        "brain natriuretic peptide"
    ],

    "lactate": [
        "lactate",
        "lactic acid"
    ],

    "troponin": [
        "troponin - i",
        "troponin i",
        "troponin-i",
        "troponin - t",
        "troponin t",
        "troponin-t"
    ],

    "sodium": [
        "sodium"
    ],

    "potassium": [
        "potassium"
    ],

    "chloride": [
        "chloride"
    ],

    "bicarbonate": [
        "bicarbonate",
        "hco3"
    ],

    "bun": [
        "bun",
        "blood urea nitrogen"
    ],

    "glucose": [
        "glucose"
    ],

    "hemoglobin": [
        "hgb",
        "hemoglobin"
    ],

    "wbc": [
        "wbc x 1000",
        "wbc",
        "white blood cell count"
    ],

    "platelets": [
        "platelets x 1000",
        "platelets",
        "platelet count"
    ],

    "inr": [
        "inr"
    ],

    "pt": [
        "pt"
    ],

    "ptt": [
        "ptt",
        "aptt"
    ],

    "ph": [
        "ph"
    ],

    "pao2": [
        "pao2"
    ],

    "paco2": [
        "paco2"
    ],

    "fio2": [
        "fio2"
    ],

    "albumin": [
        "albumin"
    ],

    "bilirubin_total": [
        "total bilirubin",
        "bilirubin total",
        "bilirubin, total"
    ],

    "alt": [
        "alt (sgpt)",
        "alt",
        "sgpt"
    ],

    "ast": [
        "ast (sgot)",
        "ast",
        "sgot"
    ]
}


# Reverse mapping
alias_to_feature = {}

for feature, aliases in LAB_ALIASES.items():
    for alias in aliases:
        alias_to_feature[
            alias.lower().strip()
        ] = feature


lab["feature"] = (
    lab["labname_clean"]
    .map(alias_to_feature)
)


mapped_lab = lab[
    lab["feature"].notna() &
    lab["labresult"].notna()
].copy()

print(
    f"✓ Mapped lab rows: "
    f"{len(mapped_lab):,}"
)

print(
    f"✓ Patients with mapped labs: "
    f"{mapped_lab['patientunitstayid'].nunique():,}"
)


# ------------------------------------------------------------
# 5. HOURLY MEDIAN
# ------------------------------------------------------------

lab_hour_long = (
    mapped_lab
    .groupby(
        [
            "patientunitstayid",
            "hour",
            "feature"
        ],
        as_index=False
    )["labresult"]
    .median()
)


lab_hour = (
    lab_hour_long
    .pivot(
        index=[
            "patientunitstayid",
            "hour"
        ],
        columns="feature",
        values="labresult"
    )
    .reset_index()
)

lab_hour.columns.name = None


# ------------------------------------------------------------
# 6. EXPECTED COST LAB FEATURES
# ------------------------------------------------------------

LAB_FEATURES = [
    "creatinine",
    "bnp",
    "lactate",
    "troponin",
    "sodium",
    "potassium",
    "chloride",
    "bicarbonate",
    "bun",
    "glucose",
    "hemoglobin",
    "wbc",
    "platelets",
    "inr",
    "pt",
    "ptt",
    "ph",
    "pao2",
    "paco2",
    "fio2",
    "albumin",
    "bilirubin_total",
    "alt",
    "ast"
]

for f in LAB_FEATURES:
    if f not in lab_hour.columns:
        lab_hour[f] = np.nan

lab_hour = lab_hour[
    ["patientunitstayid", "hour"] +
    LAB_FEATURES
].copy()


# ------------------------------------------------------------
# 7. REPORT
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("LAB FEATURE AVAILABILITY")
print("=" * 85)

for f in LAB_FEATURES:

    n = lab_hour[f].notna().sum()

    pct = (
        lab_hour[f].notna().mean()
        * 100
    )

    patients = lab_hour.loc[
        lab_hour[f].notna(),
        "patientunitstayid"
    ].nunique()

    print(
        f"{f:20s}: "
        f"{n:8,d} "
        f"({pct:6.2f}%) | "
        f"patients={patients:,}"
    )


print("\n" + "=" * 85)
print("LAB MAPPING SUMMARY")
print("=" * 85)

print(
    f"Hourly lab rows : "
    f"{len(lab_hour):,}"
)

print(
    f"Patients        : "
    f"{lab_hour['patientunitstayid'].nunique():,}"
)

available = [
    f for f in LAB_FEATURES
    if lab_hour[f].notna().any()
]

unavailable = [
    f for f in LAB_FEATURES
    if not lab_hour[f].notna().any()
]

print(
    f"Available features: "
    f"{len(available)}/{len(LAB_FEATURES)}"
)

print("\nAvailable:")
print(available)

print("\nUnavailable:")
print(unavailable)

print("\n" + "=" * 85)
print("✓ LAB MAPPING COMPLETE")
print("✓ 0–48 h WINDOW")
print("✓ NO IMPUTATION")
print("✓ NO NORMALIZATION")
print("✓ NO TRAINING")
print("=" * 85)

# Memory cleanup
del lab
del mapped_lab
del lab_hour_long
gc.collect()

In [ ]:
# ============================================================
# CELL 8B — AUDIT eICU INR / COAGULATION LAB NAMES
# READ-ONLY — NO TRAINING
# ============================================================

import pandas as pd

print("=" * 85)
print("AUDIT eICU INR / COAGULATION LAB NAMES")
print("=" * 85)

lab_path = EICU_DIR / "lab.csv.gz"

# Read only labname -> much lighter than loading full table
lab_names = pd.read_csv(
    lab_path,
    compression="gzip",
    usecols=["labname"],
    low_memory=False
)

lab_names["labname_clean"] = (
    lab_names["labname"]
    .astype(str)
    .str.lower()
    .str.strip()
)

unique_names = sorted(
    lab_names["labname_clean"]
    .dropna()
    .unique()
)

patterns = [
    "inr",
    "prothrombin",
    "protime",
    "pro time",
    "pt ",
    "pt-",
    "ptt",
    "aptt"
]

matches = [
    name
    for name in unique_names
    if any(p in name for p in patterns)
]

print("\nPotential coagulation lab names found:\n")

for name in matches:
    print(" -", name)

print("\nTotal matches:", len(matches))

del lab_names

print("\n✓ AUDIT COMPLETE")

In [ ]:
# ============================================================
# CELL 8B — AUDIT eICU INR / COAGULATION LAB NAMES
# READ-ONLY — NO TRAINING
# ============================================================

import pandas as pd

print("=" * 85)
print("AUDIT eICU INR / COAGULATION LAB NAMES")
print("=" * 85)

lab_path = EICU_DIR / "lab.csv.gz"

# Read only labname -> much lighter than loading full table
lab_names = pd.read_csv(
    lab_path,
    compression="gzip",
    usecols=["labname"],
    low_memory=False
)

lab_names["labname_clean"] = (
    lab_names["labname"]
    .astype(str)
    .str.lower()
    .str.strip()
)

unique_names = sorted(
    lab_names["labname_clean"]
    .dropna()
    .unique()
)

patterns = [
    "inr",
    "prothrombin",
    "protime",
    "pro time",
    "pt ",
    "pt-",
    "ptt",
    "aptt"
]

matches = [
    name
    for name in unique_names
    if any(p in name for p in patterns)
]

print("\nPotential coagulation lab names found:\n")

for name in matches:
    print(" -", name)

print("\nTotal matches:", len(matches))

del lab_names

print("\n✓ AUDIT COMPLETE")

In [ ]:
# ============================================================
# CELL 8C — FINAL LAB MAPPING WITH CORRECTED INR
# Rebuild lab_hour efficiently
# NO TRAINING
# ============================================================

import pandas as pd
import numpy as np
import gc

print("=" * 85)
print("FINAL eICU LAB MAPPING — CORRECTED INR")
print("=" * 85)

lab_path = EICU_DIR / "lab.csv.gz"

# Only columns actually needed
usecols = [
    "patientunitstayid",
    "labresultoffset",
    "labname",
    "labresult"
]

print("Reading required lab columns only ...")

lab = pd.read_csv(
    lab_path,
    compression="gzip",
    usecols=usecols,
    low_memory=False
)

# ------------------------------------------------------------
# Cohort + 0–48 h
# ------------------------------------------------------------

lab["patientunitstayid"] = pd.to_numeric(
    lab["patientunitstayid"],
    errors="coerce"
)

lab["labresultoffset"] = pd.to_numeric(
    lab["labresultoffset"],
    errors="coerce"
)

lab = lab[
    lab["patientunitstayid"].isin(ids) &
    (lab["labresultoffset"] >= 0) &
    (lab["labresultoffset"] < 48 * 60)
].copy()

lab["hour"] = (
    lab["labresultoffset"] // 60
).astype(int)

lab["labresult"] = pd.to_numeric(
    lab["labresult"],
    errors="coerce"
)

lab["labname_clean"] = (
    lab["labname"]
    .astype(str)
    .str.lower()
    .str.strip()
)

print(f"✓ Eligible 0–48 h rows: {len(lab):,}")


# ------------------------------------------------------------
# FINAL VERIFIED eICU -> COST mapping
# ------------------------------------------------------------

LAB_ALIASES = {

    "creatinine": ["creatinine"],

    "bnp": [
        "bnp",
        "brain natriuretic peptide"
    ],

    "lactate": [
        "lactate",
        "lactic acid"
    ],

    "troponin": [
        "troponin - i",
        "troponin i",
        "troponin-i",
        "troponin - t",
        "troponin t",
        "troponin-t"
    ],

    "sodium": ["sodium"],

    "potassium": ["potassium"],

    "chloride": ["chloride"],

    "bicarbonate": [
        "bicarbonate",
        "hco3"
    ],

    "bun": [
        "bun",
        "blood urea nitrogen"
    ],

    "glucose": ["glucose"],

    "hemoglobin": [
        "hgb",
        "hemoglobin"
    ],

    "wbc": [
        "wbc x 1000",
        "wbc",
        "white blood cell count"
    ],

    "platelets": [
        "platelets x 1000",
        "platelets",
        "platelet count"
    ],

    # CORRECTED
    "inr": [
        "pt - inr"
    ],

    "pt": [
        "pt"
    ],

    "ptt": [
        "ptt",
        "aptt"
    ],

    "ph": ["ph"],

    "pao2": ["pao2"],

    "paco2": ["paco2"],

    "fio2": ["fio2"],

    "albumin": ["albumin"],

    "bilirubin_total": [
        "total bilirubin",
        "bilirubin total",
        "bilirubin, total"
    ],

    "alt": [
        "alt (sgpt)",
        "alt",
        "sgpt"
    ],

    "ast": [
        "ast (sgot)",
        "ast",
        "sgot"
    ]
}


alias_to_feature = {
    alias.lower().strip(): feature
    for feature, aliases in LAB_ALIASES.items()
    for alias in aliases
}

lab["feature"] = (
    lab["labname_clean"]
    .map(alias_to_feature)
)

mapped_lab = lab[
    lab["feature"].notna() &
    lab["labresult"].notna()
].copy()

print(
    f"✓ Mapped rows: "
    f"{len(mapped_lab):,}"
)


# ------------------------------------------------------------
# Hourly median
# ------------------------------------------------------------

lab_hour_long = (
    mapped_lab
    .groupby(
        [
            "patientunitstayid",
            "hour",
            "feature"
        ],
        as_index=False
    )["labresult"]
    .median()
)

lab_hour = (
    lab_hour_long
    .pivot(
        index=[
            "patientunitstayid",
            "hour"
        ],
        columns="feature",
        values="labresult"
    )
    .reset_index()
)

lab_hour.columns.name = None


LAB_FEATURES = [
    "creatinine",
    "bnp",
    "lactate",
    "troponin",
    "sodium",
    "potassium",
    "chloride",
    "bicarbonate",
    "bun",
    "glucose",
    "hemoglobin",
    "wbc",
    "platelets",
    "inr",
    "pt",
    "ptt",
    "ph",
    "pao2",
    "paco2",
    "fio2",
    "albumin",
    "bilirubin_total",
    "alt",
    "ast"
]

for f in LAB_FEATURES:
    if f not in lab_hour.columns:
        lab_hour[f] = np.nan

lab_hour = lab_hour[
    ["patientunitstayid", "hour"] +
    LAB_FEATURES
].copy()


# ------------------------------------------------------------
# FINAL COVERAGE
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("FINAL LAB FEATURE AVAILABILITY")
print("=" * 85)

for f in LAB_FEATURES:

    n = lab_hour[f].notna().sum()

    patients = lab_hour.loc[
        lab_hour[f].notna(),
        "patientunitstayid"
    ].nunique()

    print(
        f"{f:20s}: "
        f"{n:8,d} | "
        f"patients={patients:,}"
    )


print("\n" + "=" * 85)

available = [
    f for f in LAB_FEATURES
    if lab_hour[f].notna().any()
]

print(
    f"Available COST lab features: "
    f"{len(available)}/{len(LAB_FEATURES)}"
)

print(
    f"Hourly lab rows: "
    f"{len(lab_hour):,}"
)

print(
    f"Patients: "
    f"{lab_hour['patientunitstayid'].nunique():,}"
)

print("=" * 85)
print("✓ FINAL LAB MAPPING COMPLETE")
print("✓ INR CORRECTED: 'pt - inr' -> inr")
print("✓ NO IMPUTATION")
print("✓ NO NORMALIZATION")
print("✓ NO TRAINING")
print("=" * 85)

del lab
del mapped_lab
del lab_hour_long

gc.collect()

In [ ]:
# ============================================================
# CELL 9 — BUILD FINAL 48h × 31 TEMPORAL eICU GRID
# Merge periodic vitals + laboratory features
# NO IMPUTATION
# NO NORMALIZATION
# NO TRAINING
# ============================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("BUILDING eICU 48h × 31 TEMPORAL FEATURE GRID")
print("=" * 90)

# ------------------------------------------------------------
# 1. DEFINITIONS
# ------------------------------------------------------------

VITAL_SIGNS = [
    "heart_rate",
    "sbp",
    "dbp",
    "map",
    "resp_rate",
    "spo2",
    "temperature"
]

LAB_FEATURES = [
    "creatinine",
    "bnp",
    "lactate",
    "troponin",
    "sodium",
    "potassium",
    "chloride",
    "bicarbonate",
    "bun",
    "glucose",
    "hemoglobin",
    "wbc",
    "platelets",
    "inr",
    "pt",
    "ptt",
    "ph",
    "pao2",
    "paco2",
    "fio2",
    "albumin",
    "bilirubin_total",
    "alt",
    "ast"
]

TEMPORAL_FEATURES = VITAL_SIGNS + LAB_FEATURES

assert len(VITAL_SIGNS) == 7
assert len(LAB_FEATURES) == 24
assert len(TEMPORAL_FEATURES) == 31

print("Vital features :", len(VITAL_SIGNS))
print("Lab features   :", len(LAB_FEATURES))
print("Total temporal :", len(TEMPORAL_FEATURES))


# ------------------------------------------------------------
# 2. CHECK INPUT TABLES
# ------------------------------------------------------------

required_vital_cols = (
    ["patientunitstayid", "hour"] +
    VITAL_SIGNS
)

required_lab_cols = (
    ["patientunitstayid", "hour"] +
    LAB_FEATURES
)

missing_vital = [
    c for c in required_vital_cols
    if c not in vital_hour.columns
]

missing_lab = [
    c for c in required_lab_cols
    if c not in lab_hour.columns
]

if missing_vital:
    raise ValueError(
        f"Missing vital columns: {missing_vital}"
    )

if missing_lab:
    raise ValueError(
        f"Missing lab columns: {missing_lab}"
    )

print("\n✓ vital_hour structure OK")
print("✓ lab_hour structure OK")


# ------------------------------------------------------------
# 3. CREATE COMPLETE 48-HOUR GRID
# ------------------------------------------------------------

cohort_ids = (
    cohort["patientunitstayid"]
    .astype(int)
    .drop_duplicates()
    .sort_values()
    .to_numpy()
)

print(
    f"\nCohort patients: "
    f"{len(cohort_ids):,}"
)

expected_rows = (
    len(cohort_ids) * 48
)

grid = pd.MultiIndex.from_product(
    [
        cohort_ids,
        np.arange(48, dtype=int)
    ],
    names=[
        "patientunitstayid",
        "hour"
    ]
).to_frame(index=False)

print(
    f"Expected patient-hour rows: "
    f"{expected_rows:,}"
)

print(
    f"Generated grid rows       : "
    f"{len(grid):,}"
)

assert len(grid) == expected_rows


# ------------------------------------------------------------
# 4. MERGE PERIODIC VITALS
# ------------------------------------------------------------

temporal = grid.merge(
    vital_hour[
        ["patientunitstayid", "hour"] +
        VITAL_SIGNS
    ],
    on=[
        "patientunitstayid",
        "hour"
    ],
    how="left",
    validate="one_to_one"
)

print(
    f"\n✓ Vitals merged: "
    f"{len(temporal):,} rows"
)


# ------------------------------------------------------------
# 5. MERGE LABS
# ------------------------------------------------------------

temporal = temporal.merge(
    lab_hour[
        ["patientunitstayid", "hour"] +
        LAB_FEATURES
    ],
    on=[
        "patientunitstayid",
        "hour"
    ],
    how="left",
    validate="one_to_one"
)

print(
    f"✓ Labs merged  : "
    f"{len(temporal):,} rows"
)

assert len(temporal) == expected_rows


# ------------------------------------------------------------
# 6. FORCE FINAL COLUMN ORDER
# ------------------------------------------------------------

temporal = temporal[
    [
        "patientunitstayid",
        "hour"
    ] +
    TEMPORAL_FEATURES
].copy()

assert temporal.shape[1] == 33

print(
    f"✓ Final columns: "
    f"{temporal.shape[1]} "
    f"(ID + hour + 31 features)"
)


# ------------------------------------------------------------
# 7. DUPLICATE / RANGE AUDIT
# ------------------------------------------------------------

duplicates = temporal.duplicated(
    subset=[
        "patientunitstayid",
        "hour"
    ]
).sum()

print(
    f"\nDuplicate patient-hour rows: "
    f"{duplicates:,}"
)

print(
    f"Hour range: "
    f"{temporal['hour'].min()}–"
    f"{temporal['hour'].max()}"
)

assert duplicates == 0
assert temporal["hour"].min() == 0
assert temporal["hour"].max() == 47


# ------------------------------------------------------------
# 8. RAW COVERAGE / MISSINGNESS
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("RAW eICU TEMPORAL FEATURE COVERAGE")
print("=" * 90)

coverage_records = []

for f in TEMPORAL_FEATURES:

    observed = int(
        temporal[f]
        .notna()
        .sum()
    )

    total = len(temporal)

    coverage = (
        observed / total * 100
    )

    missing = (
        100 - coverage
    )

    patient_count = int(
        temporal.loc[
            temporal[f].notna(),
            "patientunitstayid"
        ].nunique()
    )

    coverage_records.append({
        "feature": f,
        "observed_values": observed,
        "coverage_pct": coverage,
        "missing_pct": missing,
        "patients_observed": patient_count
    })

    print(
        f"{f:20s}: "
        f"coverage={coverage:6.2f}% | "
        f"missing={missing:6.2f}% | "
        f"patients={patient_count:,}"
    )


coverage_df = pd.DataFrame(
    coverage_records
)


# ------------------------------------------------------------
# 9. PATIENT-LEVEL DATA AVAILABILITY
# ------------------------------------------------------------

temporal["observed_feature_count"] = (
    temporal[TEMPORAL_FEATURES]
    .notna()
    .sum(axis=1)
)

patient_availability = (
    temporal
    .groupby("patientunitstayid")
    .agg(
        observed_values=(
            "observed_feature_count",
            "sum"
        ),
        hours_with_any_data=(
            "observed_feature_count",
            lambda x: int((x > 0).sum())
        )
    )
    .reset_index()
)

max_values_per_patient = (
    48 * len(TEMPORAL_FEATURES)
)

patient_availability[
    "overall_coverage_pct"
] = (
    patient_availability[
        "observed_values"
    ]
    / max_values_per_patient
    * 100
)


# ------------------------------------------------------------
# 10. SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("TEMPORAL GRID SUMMARY")
print("=" * 90)

print(
    f"Patients             : "
    f"{len(cohort_ids):,}"
)

print(
    f"Hours per patient    : 48"
)

print(
    f"Temporal features    : "
    f"{len(TEMPORAL_FEATURES)}"
)

print(
    f"Patient-hour rows    : "
    f"{len(temporal):,}"
)

print(
    f"Expected tensor shape: "
    f"({len(cohort_ids):,}, 48, 31)"
)

print(
    "\nPatient-level raw coverage:"
)

print(
    patient_availability[
        "overall_coverage_pct"
    ].describe(
        percentiles=[
            .25, .50, .75
        ]
    )
)

print(
    "\nHours with any observed data:"
)

print(
    patient_availability[
        "hours_with_any_data"
    ].describe(
        percentiles=[
            .25, .50, .75
        ]
    )
)


# Remove temporary helper column
temporal.drop(
    columns=["observed_feature_count"],
    inplace=True
)


print("\n" + "=" * 90)
print("✓ 48 × 31 TEMPORAL GRID COMPLETE")
print("✓ RAW MISSINGNESS PRESERVED")
print("✓ NO IMPUTATION")
print("✓ NO NORMALIZATION")
print("✓ NO TRAINING")
print("=" * 90)

In [ ]:
# ============================================================
# CELL 10 — BUILD 11 STATIC FEATURES FOR eICU
# NO TRAINING
# NO NORMALIZATION
# ============================================================

import pandas as pd
import numpy as np
import re

print("=" * 90)
print("BUILDING eICU STATIC FEATURES")
print("=" * 90)

STATIC_FEATURES = [
    "age",
    "gender_male",
    "admission_emergency",
    "admission_urgent",
    "elixhauser_score",
    "hypertension",
    "diabetes",
    "ckd",
    "copd",
    "af",
    "prior_hf"
]

# ------------------------------------------------------------
# 1. START FROM EXACT EXTERNAL COHORT
# ------------------------------------------------------------

static = cohort[
    ["patientunitstayid"]
].drop_duplicates().copy()

print(f"Cohort patients: {len(static):,}")


# ------------------------------------------------------------
# 2. AGE
# eICU may encode >89 as "> 89"
# ------------------------------------------------------------

p = patient[
    [
        "patientunitstayid",
        "age",
        "gender",
        "hospitaladmitsource",
        "unitadmitsource",
        "apacheadmissiondx"
    ]
].copy()

def parse_age(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    if s in ["> 89", ">89"]:
        return 90.0

    try:
        return float(s)
    except:
        return np.nan

p["age"] = p["age"].apply(parse_age)


# ------------------------------------------------------------
# 3. GENDER
# ------------------------------------------------------------

p["gender_male"] = (
    p["gender"]
    .astype(str)
    .str.lower()
    .str.strip()
    .eq("male")
    .astype(float)
)

p.loc[
    p["gender"].isna(),
    "gender_male"
] = np.nan


# ------------------------------------------------------------
# 4. ADMISSION TYPE PROXIES
# ------------------------------------------------------------

source_text = (
    p["hospitaladmitsource"]
    .fillna("")
    .astype(str)
    + " " +
    p["unitadmitsource"]
    .fillna("")
    .astype(str)
).str.lower()

p["admission_emergency"] = (
    source_text.str.contains(
        r"emergency|emergency department|ed",
        regex=True
    )
).astype(float)

p["admission_urgent"] = (
    source_text.str.contains(
        r"urgent",
        regex=True
    )
).astype(float)


static = static.merge(
    p[
        [
            "patientunitstayid",
            "age",
            "gender_male",
            "admission_emergency",
            "admission_urgent"
        ]
    ],
    on="patientunitstayid",
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# 5. COMORBIDITIES
# Combine diagnosis + past history text
# ------------------------------------------------------------

dx = diagnosis[
    [
        "patientunitstayid",
        "diagnosisstring",
        "icd9code"
    ]
].copy()

dx["text"] = (
    dx["diagnosisstring"]
    .fillna("")
    .astype(str)
    + " " +
    dx["icd9code"]
    .fillna("")
    .astype(str)
).str.lower()


ph = pasthistory[
    [
        "patientunitstayid",
        "pasthistorypath",
        "pasthistoryvalue",
        "pasthistoryvaluetext"
    ]
].copy()

ph["text"] = (
    ph["pasthistorypath"]
    .fillna("")
    .astype(str)
    + " " +
    ph["pasthistoryvalue"]
    .fillna("")
    .astype(str)
    + " " +
    ph["pasthistoryvaluetext"]
    .fillna("")
    .astype(str)
).str.lower()


dx_text = (
    dx.groupby("patientunitstayid")["text"]
    .apply(" ".join)
)

ph_text = (
    ph.groupby("patientunitstayid")["text"]
    .apply(" ".join)
)

clinical_text = (
    pd.concat(
        [dx_text, ph_text],
        axis=1
    )
    .fillna("")
)

clinical_text["all_text"] = (
    clinical_text
    .astype(str)
    .agg(" ".join, axis=1)
)


# ------------------------------------------------------------
# 6. PRESPECIFIED TEXT/ICD PHENOTYPES
# ------------------------------------------------------------

patterns = {

    "hypertension":
        r"hypertension|hypertensive|\b401(?:\.|\b)",

    "diabetes":
        r"diabetes|diabetic|\b250(?:\.|\b)",

    "ckd":
        r"chronic kidney|chronic renal|renal failure|"
        r"\b585(?:\.|\b)",

    "copd":
        r"copd|chronic obstructive|emphysema|"
        r"\b496(?:\.|\b)",

    "af":
        r"atrial fibrillation|atrial fib|\b427\.31\b",

    "prior_hf":
        r"heart failure|congestive heart failure|\bchf\b|"
        r"\b428(?:\.|\b)"
}


comorbidity = pd.DataFrame(
    index=clinical_text.index
)

for feature, pattern in patterns.items():

    comorbidity[feature] = (
        clinical_text["all_text"]
        .str.contains(
            pattern,
            regex=True,
            na=False
        )
        .astype(float)
    )

comorbidity = (
    comorbidity
    .reset_index()
)


static = static.merge(
    comorbidity,
    on="patientunitstayid",
    how="left",
    validate="one_to_one"
)

for f in [
    "hypertension",
    "diabetes",
    "ckd",
    "copd",
    "af",
    "prior_hf"
]:
    static[f] = static[f].fillna(0.0)


# ------------------------------------------------------------
# 7. ELIXHAUSER
# ------------------------------------------------------------
# The current harmonized eICU input does NOT contain the same
# precomputed Elixhauser score used by the MIMIC-IV pipeline.
#
# Do NOT invent/recalculate an incompatible score here.
# Keep it explicitly unavailable for the compatibility audit.
# Handling for model inference will be defined using the
# MIMIC training preprocessing parameters in a later cell.

static["elixhauser_score"] = np.nan


# ------------------------------------------------------------
# 8. FINAL ORDER
# ------------------------------------------------------------

static = static[
    ["patientunitstayid"] +
    STATIC_FEATURES
].copy()

assert len(static) == len(cohort)


# ------------------------------------------------------------
# 9. AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("STATIC FEATURE AUDIT")
print("=" * 90)

for f in STATIC_FEATURES:

    n = static[f].notna().sum()
    pct = static[f].notna().mean() * 100

    if f in [
        "gender_male",
        "admission_emergency",
        "admission_urgent",
        "hypertension",
        "diabetes",
        "ckd",
        "copd",
        "af",
        "prior_hf"
    ]:
        positive = (
            static[f].fillna(0) == 1
        ).sum()

        print(
            f"{f:22s}: "
            f"available={pct:6.2f}% | "
            f"positive={positive:,}"
        )

    else:
        print(
            f"{f:22s}: "
            f"available={pct:6.2f}%"
        )


print("\n" + "=" * 90)
print("STATIC TABLE SUMMARY")
print("=" * 90)

print(
    f"Patients       : "
    f"{len(static):,}"
)

print(
    f"Static features: "
    f"{len(STATIC_FEATURES)}"
)

print(
    f"Expected shape : "
    f"({len(cohort):,}, 11)"
)

print(
    f"Actual shape   : "
    f"({len(static):,}, "
    f"{len(STATIC_FEATURES)})"
)

print("\n✓ STATIC FEATURE TABLE COMPLETE")
print("✓ ELIXHAUSER EXPLICITLY MARKED UNAVAILABLE")
print("✓ NO IMPUTATION")
print("✓ NO NORMALIZATION")
print("✓ NO TRAINING")
print("=" * 90)

In [ ]:
# ============================================================
# CELL 11 — AUDIT COST IMPROVED CHECKPOINT
# External validation preparation
# READ ONLY — NO TRAINING
# ============================================================

import torch
from pathlib import Path

print("=" * 90)
print("COST IMPROVED CHECKPOINT AUDIT")
print("=" * 90)

print("\nCheckpoint:")
print(CKPT_PATH)

if not CKPT_PATH.exists():
    raise FileNotFoundError(
        f"COST checkpoint not found:\n{CKPT_PATH}"
    )

print(
    f"\n✓ Checkpoint exists "
    f"({CKPT_PATH.stat().st_size / 1024**2:.2f} MB)"
)

# ------------------------------------------------------------
# LOAD CHECKPOINT ON CPU
# ------------------------------------------------------------

try:
    ckpt = torch.load(
        CKPT_PATH,
        map_location="cpu",
        weights_only=False
    )
except TypeError:
    # compatibility with older PyTorch
    ckpt = torch.load(
        CKPT_PATH,
        map_location="cpu"
    )

print("\n✓ Checkpoint loaded successfully")

print("\nObject type:")
print(type(ckpt))


# ------------------------------------------------------------
# TOP-LEVEL CONTENT
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CHECKPOINT CONTENT")
print("=" * 90)

if isinstance(ckpt, dict):

    print("\nTop-level keys:")

    for k in ckpt.keys():
        obj = ckpt[k]

        if hasattr(obj, "shape"):
            print(
                f"  {k:30s} "
                f"shape={tuple(obj.shape)}"
            )
        else:
            print(
                f"  {k:30s} "
                f"type={type(obj).__name__}"
            )

else:
    print(
        "\nCheckpoint is not a dictionary."
    )


# ------------------------------------------------------------
# SEARCH FOR PREPROCESSING INFORMATION
# ------------------------------------------------------------

keywords = [
    "mean",
    "std",
    "median",
    "scale",
    "scaler",
    "imputer",
    "feature",
    "temporal",
    "static",
    "normal",
    "norm",
    "input"
]

print("\n" + "=" * 90)
print("POTENTIAL PREPROCESSING / FEATURE INFORMATION")
print("=" * 90)

found = False

if isinstance(ckpt, dict):

    for key, value in ckpt.items():

        key_lower = str(key).lower()

        if any(
            word in key_lower
            for word in keywords
        ):
            found = True

            print(
                f"\nKEY: {key}"
            )

            if isinstance(
                value,
                (list, tuple)
            ):
                print(
                    f"Length: {len(value)}"
                )
                print(value)

            elif isinstance(
                value,
                dict
            ):
                print(
                    "Subkeys:",
                    list(value.keys())[:50]
                )

            elif hasattr(
                value,
                "shape"
            ):
                print(
                    "Shape:",
                    tuple(value.shape)
                )

                try:
                    print(value)
                except:
                    pass

            else:
                print(value)


if not found:
    print(
        "\nNo obvious preprocessing metadata "
        "stored in checkpoint."
    )


# ------------------------------------------------------------
# IDENTIFY STATE DICT
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("MODEL STATE DICTIONARY")
print("=" * 90)

state_dict = None

if isinstance(ckpt, dict):

    candidate_keys = [
        "model_state_dict",
        "state_dict",
        "model",
        "model_state"
    ]

    for k in candidate_keys:

        if (
            k in ckpt
            and isinstance(ckpt[k], dict)
        ):
            state_dict = ckpt[k]

            print(
                f"✓ State dictionary found under: "
                f"'{k}'"
            )

            break

    # checkpoint itself may be a state_dict
    if state_dict is None:

        tensor_values = [
            isinstance(v, torch.Tensor)
            for v in ckpt.values()
        ]

        if (
            len(tensor_values) > 0
            and all(tensor_values)
        ):
            state_dict = ckpt

            print(
                "✓ Checkpoint itself appears "
                "to be a state_dict"
            )


if state_dict is not None:

    print(
        f"Number of model tensors: "
        f"{len(state_dict):,}"
    )

    print("\nFirst model parameters:")

    for i, (name, tensor) in enumerate(
        state_dict.items()
    ):

        print(
            f"  {name:50s} "
            f"{tuple(tensor.shape)}"
        )

        if i >= 19:
            break

else:

    print(
        "⚠ Could not automatically identify "
        "model state_dict."
    )


# ------------------------------------------------------------
# EXTERNAL VALIDATION STATUS
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("EXTERNAL VALIDATION STATUS")
print("=" * 90)

print("eICU HF cohort       : 12,317")
print("Temporal dimensions  : 48 × 31")
print("Static dimensions    : 11")
print("COST checkpoint      : FOUND")
print("Training on eICU     : NO")

print("\nPending before inference:")
print("  1. Recover original MIMIC preprocessing parameters")
print("  2. Verify exact COST model architecture")
print("  3. Harmonize unavailable static features")
print("  4. Apply MIMIC-trained imputation/normalization")
print("  5. Apply prespecified minimum-data eligibility")
print("  6. Load frozen COST Improved weights")
print("  7. Run eICU inference")
print("  8. Compute AUROC/AUPRC/sensitivity/specificity/etc.")

print("\n✓ CHECKPOINT AUDIT COMPLETE")
print("✓ NO TRAINING PERFORMED")
print("=" * 90)

In [ ]:
# ============================================================
# CELL 12 — AUDIT COST CONFIG + MIMIC TRAINING NORM STATS
# READ ONLY
# NO TRAINING
# ============================================================

import numpy as np
import torch

print("=" * 90)
print("COST CONFIGURATION + NORMALIZATION AUDIT")
print("=" * 90)

# ------------------------------------------------------------
# 1. CONFIG
# ------------------------------------------------------------

config = ckpt.get("config", {})

print("\nCONFIG")
print("-" * 90)

if isinstance(config, dict):
    for key, value in config.items():
        print(f"{key:30s}: {value}")
else:
    print("Config type:", type(config))
    print(config)


# ------------------------------------------------------------
# 2. NORMALIZATION STATS
# ------------------------------------------------------------

norm_stats = ckpt.get("norm_stats", {})

print("\n" + "=" * 90)
print("MIMIC-IV TRAINING NORMALIZATION STATISTICS")
print("=" * 90)

for key in [
    "feat_mean",
    "feat_std",
    "stat_mean",
    "stat_std"
]:
    value = norm_stats.get(key)

    print(f"\n{key}")

    if value is None:
        print("  MISSING")
        continue

    if torch.is_tensor(value):
        arr = value.detach().cpu().numpy()
    else:
        arr = np.asarray(value)

    print("  type  :", type(value).__name__)
    print("  shape :", arr.shape)
    print("  values:")
    print(arr)


# ------------------------------------------------------------
# 3. DIMENSION CHECK
# ------------------------------------------------------------

def to_numpy(x):
    if x is None:
        return None
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)


feat_mean = to_numpy(norm_stats.get("feat_mean"))
feat_std  = to_numpy(norm_stats.get("feat_std"))

stat_mean = to_numpy(norm_stats.get("stat_mean"))
stat_std  = to_numpy(norm_stats.get("stat_std"))


print("\n" + "=" * 90)
print("DIMENSION COMPATIBILITY")
print("=" * 90)

if feat_mean is not None:
    print(
        "Temporal mean elements :",
        feat_mean.size
    )

if feat_std is not None:
    print(
        "Temporal std elements  :",
        feat_std.size
    )

if stat_mean is not None:
    print(
        "Static mean elements   :",
        stat_mean.size
    )

if stat_std is not None:
    print(
        "Static std elements    :",
        stat_std.size
    )

print("\nExpected:")
print("  Temporal raw features : 31")
print("  Static features       : 11")

if feat_mean is not None:
    print(
        "\nTemporal stats compatible:",
        feat_mean.size == 31
    )

if stat_mean is not None:
    print(
        "Static stats compatible  :",
        stat_mean.size == 11
    )


# ------------------------------------------------------------
# 4. INPUT PROJECTION AUDIT
# ------------------------------------------------------------

state_dict = ckpt["model_state_dict"]

input_weight = state_dict.get(
    "input_proj.0.weight"
)

print("\n" + "=" * 90)
print("MODEL INPUT DIMENSION")
print("=" * 90)

if input_weight is not None:

    model_input_dim = input_weight.shape[1]

    print(
        "input_proj input dimension :",
        model_input_dim
    )

    print(
        "31 raw temporal features   :",
        len(TEMPORAL_FEATURES)
    )

    print(
        "Input/raw ratio            :",
        model_input_dim /
        len(TEMPORAL_FEATURES)
    )

else:
    print(
        "input_proj.0.weight not found"
    )


# ------------------------------------------------------------
# 5. CHECK STORED PERFORMANCE
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("STORED ORIGINAL MODEL PERFORMANCE")
print("=" * 90)

for key in [
    "val_auc",
    "test_auc_single",
    "test_auc_ensemble"
]:

    value = ckpt.get(key)

    if value is None:
        print(f"{key:25s}: unavailable")
    else:

        if torch.is_tensor(value):
            value = value.item()

        print(
            f"{key:25s}: "
            f"{float(value):.6f}"
        )


# ------------------------------------------------------------
# 6. IMPORTANT EXTERNAL VALIDATION CHECK
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("NEXT-STEP READINESS")
print("=" * 90)

print("Temporal eICU grid      : READY (48 × 31)")
print("Static eICU table       : READY (11)")
print("Frozen COST checkpoint  : READY")
print("MIMIC normalization     : FOUND")
print("eICU retraining         : NONE")

print("\nStill need to establish:")
print("  • exact meaning/order of the 3 × 31 temporal input channels")
print("  • exact missing-value/imputation logic used during COST training")
print("  • treatment of unavailable eICU Elixhauser score")
print("  • exact static feature order expected by checkpoint")

print("\n✓ AUDIT COMPLETE")
print("✓ NO EXTERNAL INFERENCE YET")
print("✓ NO TRAINING PERFORMED")
print("=" * 90)

In [ ]:
# ============================================================
# CELL 13 — COST-ALIGNED eICU STATIC + MINIMUM-DATA FILTER
# Reproduces original COST cohort rules
# NO TRAINING
# ============================================================

import numpy as np
import pandas as pd

print("=" * 90)
print("COST-ALIGNED eICU ELIGIBILITY")
print("=" * 90)

# ------------------------------------------------------------
# 1. RECONSTRUCT COST "ELIXHAUSER" EXACTLY AS ORIGINAL CODE
# ------------------------------------------------------------
# IMPORTANT:
# In the original COST notebook, elixhauser_score is simply
# the sum of these six binary comorbidity flags.

COMORBIDITY_FEATURES = [
    "hypertension",
    "diabetes",
    "ckd",
    "copd",
    "af",
    "prior_hf"
]

static["elixhauser_score"] = (
    static[COMORBIDITY_FEATURES]
    .fillna(0)
    .sum(axis=1)
    .astype(np.float32)
)

print("\n✓ COST-compatible elixhauser_score reconstructed")

print(
    static["elixhauser_score"]
    .describe()
)


# ------------------------------------------------------------
# 2. EXACT STATIC FEATURE ORDER FROM ORIGINAL COST NOTEBOOK
# ------------------------------------------------------------

STATIC_FEATURES = [
    "age",
    "gender_male",
    "admission_emergency",
    "admission_urgent",
    "elixhauser_score",
    "hypertension",
    "diabetes",
    "ckd",
    "copd",
    "af",
    "prior_hf"
]

static = static[
    ["patientunitstayid"] +
    STATIC_FEATURES
].copy()


# ------------------------------------------------------------
# 3. EXACT TEMPORAL FEATURE ORDER FROM ORIGINAL COST
# ------------------------------------------------------------

VITAL_SIGNS = [
    "heart_rate",
    "sbp",
    "dbp",
    "map",
    "resp_rate",
    "spo2",
    "temperature"
]

LAB_FEATURES = [
    "creatinine",
    "bnp",
    "lactate",
    "troponin",
    "sodium",
    "potassium",
    "chloride",
    "bicarbonate",
    "bun",
    "glucose",
    "hemoglobin",
    "wbc",
    "platelets",
    "inr",
    "pt",
    "ptt",
    "ph",
    "pao2",
    "paco2",
    "fio2",
    "albumin",
    "bilirubin_total",
    "alt",
    "ast"
]

TEMPORAL_FEATURES = (
    VITAL_SIGNS +
    LAB_FEATURES
)

assert len(TEMPORAL_FEATURES) == 31
assert len(STATIC_FEATURES) == 11


# ------------------------------------------------------------
# 4. COUNT RAW OBSERVATIONS PER PATIENT
# ------------------------------------------------------------
# This reproduces original COST:
#
# n_vital_obs >= 6
# n_lab_obs   >= 2
#
# Count is across ALL 48 hours and features BEFORE imputation.

patient_obs = pd.DataFrame({
    "patientunitstayid":
        temporal["patientunitstayid"].unique()
})


vital_counts = (
    temporal
    .groupby("patientunitstayid")[VITAL_SIGNS]
    .count()
    .sum(axis=1)
    .rename("n_vital_obs")
)

lab_counts = (
    temporal
    .groupby("patientunitstayid")[LAB_FEATURES]
    .count()
    .sum(axis=1)
    .rename("n_lab_obs")
)


patient_obs = (
    patient_obs
    .merge(
        vital_counts,
        on="patientunitstayid",
        how="left"
    )
    .merge(
        lab_counts,
        on="patientunitstayid",
        how="left"
    )
)

patient_obs[
    ["n_vital_obs", "n_lab_obs"]
] = (
    patient_obs[
        ["n_vital_obs", "n_lab_obs"]
    ]
    .fillna(0)
)


# ------------------------------------------------------------
# 5. EXACT COST MINIMUM-DATA FILTER
# ------------------------------------------------------------

patient_obs["eligible"] = (
    (patient_obs["n_vital_obs"] >= 6) &
    (patient_obs["n_lab_obs"] >= 2)
)

eligible_ids = (
    patient_obs.loc[
        patient_obs["eligible"],
        "patientunitstayid"
    ]
    .astype(int)
    .to_numpy()
)


# ------------------------------------------------------------
# 6. AUDIT FILTER
# ------------------------------------------------------------

N_BEFORE = len(patient_obs)
N_AFTER = len(eligible_ids)

print("\n" + "=" * 90)
print("ORIGINAL COST MINIMUM-DATA FILTER")
print("=" * 90)

print(
    f"HF cohort before filter : "
    f"{N_BEFORE:,}"
)

print(
    f"Eligible after filter   : "
    f"{N_AFTER:,}"
)

print(
    f"Excluded                : "
    f"{N_BEFORE - N_AFTER:,}"
)

print(
    f"Retention               : "
    f"{N_AFTER/N_BEFORE*100:.2f}%"
)


print("\nVital observation counts:")

print(
    patient_obs[
        "n_vital_obs"
    ].describe()
)


print("\nLab observation counts:")

print(
    patient_obs[
        "n_lab_obs"
    ].describe()
)


# ------------------------------------------------------------
# 7. CREATE FINAL EXTERNAL COHORT
# ------------------------------------------------------------

cohort_external = (
    cohort[
        cohort["patientunitstayid"]
        .isin(eligible_ids)
    ]
    .copy()
)

static_external = (
    static[
        static["patientunitstayid"]
        .isin(eligible_ids)
    ]
    .copy()
)

temporal_external = (
    temporal[
        temporal["patientunitstayid"]
        .isin(eligible_ids)
    ]
    .copy()
)


# ------------------------------------------------------------
# 8. SORT IDENTICALLY
# ------------------------------------------------------------

eligible_ids_sorted = np.sort(
    eligible_ids
)

cohort_external = (
    cohort_external
    .set_index("patientunitstayid")
    .loc[eligible_ids_sorted]
    .reset_index()
)

static_external = (
    static_external
    .set_index("patientunitstayid")
    .loc[eligible_ids_sorted]
    .reset_index()
)

temporal_external = (
    temporal_external
    .sort_values(
        [
            "patientunitstayid",
            "hour"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. LABEL AUDIT
# ------------------------------------------------------------

# Find mortality label column already created in cohort.
possible_label_cols = [
    "label",
    "mortality",
    "hospital_mortality",
    "death"
]

label_col = next(
    (
        c for c in possible_label_cols
        if c in cohort_external.columns
    ),
    None
)

print("\n" + "=" * 90)
print("FINAL EXTERNAL COHORT")
print("=" * 90)

print(
    f"Patients : "
    f"{len(cohort_external):,}"
)

print(
    f"Temporal rows : "
    f"{len(temporal_external):,}"
)

print(
    "Expected temporal rows:",
    f"{len(cohort_external) * 48:,}"
)

if label_col is not None:

    y_ext = (
        cohort_external[
            label_col
        ]
        .astype(int)
        .to_numpy()
    )

    print(
        f"Deaths   : "
        f"{y_ext.sum():,}"
    )

    print(
        f"Mortality: "
        f"{y_ext.mean()*100:.2f}%"
    )

else:

    print(
        "⚠ Mortality label column "
        "not automatically identified."
    )


# ------------------------------------------------------------
# 10. FINAL CONSISTENCY
# ------------------------------------------------------------

assert (
    len(temporal_external)
    ==
    len(cohort_external) * 48
)

assert (
    len(static_external)
    ==
    len(cohort_external)
)

assert (
    static_external[
        STATIC_FEATURES
    ]
    .isna()
    .sum()
    .sum()
    <=
    static_external["gender_male"]
    .isna()
    .sum()
)


print("\nStatic feature order:")
print(STATIC_FEATURES)

print("\nTemporal feature order:")
print(TEMPORAL_FEATURES)

print("\n" + "=" * 90)
print("✓ COST MINIMUM-DATA FILTER REPRODUCED")
print("✓ COST STATIC FEATURE ORDER VERIFIED")
print("✓ COST TEMPORAL FEATURE ORDER VERIFIED")
print("✓ COST 'ELIXHAUSER' DEFINITION REPRODUCED")
print("✓ NO RETRAINING")
print("=" * 90)

In [ ]:
# ============================================================
# CELL 14 — EXTERNAL eICU PREPROCESSING FOR FROZEN COST v2
# Reproduce original COST preprocess_v2 inference behavior
# Uses MIMIC-IV normalization statistics from checkpoint
# NO TRAINING / NO FITTING ON eICU
# ============================================================

import numpy as np
import pandas as pd
import torch

print("=" * 90)
print("COST v2 EXTERNAL PREPROCESSING — eICU")
print("=" * 90)

# ------------------------------------------------------------
# 1. DEFINITIONS — EXACT COST ORDER
# ------------------------------------------------------------

VITAL_SIGNS = [
    "heart_rate", "sbp", "dbp", "map",
    "resp_rate", "spo2", "temperature"
]

LAB_FEATURES = [
    "creatinine", "bnp", "lactate", "troponin",
    "sodium", "potassium", "chloride", "bicarbonate",
    "bun", "glucose", "hemoglobin", "wbc",
    "platelets", "inr", "pt", "ptt",
    "ph", "pao2", "paco2", "fio2",
    "albumin", "bilirubin_total", "alt", "ast"
]

TEMPORAL_FEATURES = VITAL_SIGNS + LAB_FEATURES

STATIC_FEATURES = [
    "age",
    "gender_male",
    "admission_emergency",
    "admission_urgent",
    "elixhauser_score",
    "hypertension",
    "diabetes",
    "ckd",
    "copd",
    "af",
    "prior_hf"
]

N_FEATURES = 31
N_STATIC = 11
SEQ_LEN = 48

assert len(TEMPORAL_FEATURES) == 31
assert len(STATIC_FEATURES) == 11


# ------------------------------------------------------------
# 2. BUILD RAW TEMPORAL ARRAY
#    shape = N × 48 × 31
# ------------------------------------------------------------

ids_final = (
    cohort_external["patientunitstayid"]
    .astype(int)
    .to_numpy()
)

temp_sorted = (
    temporal_external
    .set_index(["patientunitstayid", "hour"])
    .loc[
        pd.MultiIndex.from_product(
            [ids_final, range(48)],
            names=["patientunitstayid", "hour"]
        )
    ]
    .reset_index()
)

X_temp_raw = (
    temp_sorted[TEMPORAL_FEATURES]
    .to_numpy(dtype=np.float32)
    .reshape(
        len(ids_final),
        SEQ_LEN,
        N_FEATURES
    )
)

print("\nRaw temporal:")
print("  shape :", X_temp_raw.shape)
print(
    "  NaNs  :",
    f"{np.isnan(X_temp_raw).sum():,}"
)


# ------------------------------------------------------------
# 3. BUILD STATIC ARRAY
# ------------------------------------------------------------

stat_sorted = (
    static_external
    .set_index("patientunitstayid")
    .loc[ids_final]
    .reset_index()
)

X_stat_raw = (
    stat_sorted[STATIC_FEATURES]
    .to_numpy(dtype=np.float32)
)

print("\nRaw static:")
print("  shape :", X_stat_raw.shape)
print(
    "  NaNs  :",
    f"{np.isnan(X_stat_raw).sum():,}"
)


# ------------------------------------------------------------
# 4. LABELS
# ------------------------------------------------------------

possible_label_cols = [
    "label",
    "mortality",
    "hospital_mortality",
    "death"
]

label_col = next(
    (
        c for c in possible_label_cols
        if c in cohort_external.columns
    ),
    None
)

if label_col is None:
    raise ValueError(
        "Mortality label column not found."
    )

y_external = (
    cohort_external[label_col]
    .astype(np.float32)
    .to_numpy()
)

print("\nLabels:")
print("  n      :", len(y_external))
print("  deaths :", int(y_external.sum()))
print(
    "  rate   :",
    f"{y_external.mean()*100:.2f}%"
)


# ------------------------------------------------------------
# 5. SAVE ORIGINAL RAW MISSINGNESS AUDIT
# ------------------------------------------------------------

raw_mask = (
    ~np.isnan(X_temp_raw)
).astype(np.float32)

print("\nOriginal raw observations:")
print(
    f"  observed : "
    f"{int(raw_mask.sum()):,}"
)

print(
    f"  missing  : "
    f"{int((1 - raw_mask).sum()):,}"
)


# ------------------------------------------------------------
# 6. REPRODUCE COST v2 LAB PREPROCESSING
# ------------------------------------------------------------
# Original preprocess_v2:
#
# lab_d = np.nan_to_num(
#     lab_d,
#     nan=0.0,
#     posinf=0.0,
#     neginf=0.0
# )
#
# Then IterativeImputer.transform().
#
# Since NaNs have already become 0, no eICU-derived
# imputation model is fitted here.
#
# We preserve that original behavior.
# ------------------------------------------------------------

X_work = X_temp_raw.copy()

n_vital = len(VITAL_SIGNS)

labs = X_work[:, :, n_vital:]

labs = np.nan_to_num(
    labs,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

X_work[:, :, n_vital:] = labs

print("\n✓ COST v2 lab missing-value behavior reproduced")


# ------------------------------------------------------------
# 7. REPRODUCE add_missingness_features()
# ------------------------------------------------------------

def add_missingness_features_external(X_temporal):

    n, seq, feats = X_temporal.shape

    # EXACTLY as COST notebook:
    # mask is calculated from array entering this function
    mask = (
        ~np.isnan(X_temporal)
    ).astype(np.float32)

    # delta-t
    delta = np.zeros_like(
        X_temporal,
        dtype=np.float32
    )

    for i in range(n):
        for k in range(feats):

            last_obs = 0

            for t in range(seq):

                if not np.isnan(
                    X_temporal[i, t, k]
                ):
                    last_obs = t
                    delta[i, t, k] = 0.0

                else:
                    delta[i, t, k] = float(
                        t - last_obs
                    )

    delta = delta / (
        seq + 1e-8
    )

    # Value channel
    values = X_temporal.copy()

    # Forward-fill ONLY vital signs
    for i in range(n):
        for k in range(n_vital):

            last = 0.0

            for t in range(seq):

                if not np.isnan(
                    values[i, t, k]
                ):
                    last = values[i, t, k]

                else:
                    values[i, t, k] = last

    # Any remaining missing -> zero
    values = np.nan_to_num(
        values,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    enhanced = np.concatenate(
        [
            values,
            mask,
            delta
        ],
        axis=-1
    )

    return enhanced.astype(
        np.float32
    )


print("\nBuilding value + mask + delta-t ...")

X_external = (
    add_missingness_features_external(
        X_work
    )
)

print(
    "✓ Enhanced temporal shape:",
    X_external.shape
)

assert X_external.shape == (
    len(ids_final),
    48,
    93
)


# ------------------------------------------------------------
# 8. LOAD FROZEN MIMIC NORMALIZATION STATS
# ------------------------------------------------------------

norm_stats = ckpt["norm_stats"]

feat_mean = np.asarray(
    norm_stats["feat_mean"],
    dtype=np.float32
)

feat_std = np.asarray(
    norm_stats["feat_std"],
    dtype=np.float32
)

stat_mean = np.asarray(
    norm_stats["stat_mean"],
    dtype=np.float32
)

stat_std = np.asarray(
    norm_stats["stat_std"],
    dtype=np.float32
)

assert feat_mean.shape == (31,)
assert feat_std.shape == (31,)
assert stat_mean.shape == (11,)
assert stat_std.shape == (11,)

print(
    "\n✓ Frozen MIMIC-IV normalization "
    "statistics loaded"
)


# ------------------------------------------------------------
# 9. NORMALIZE VALUE CHANNEL ONLY
# EXACTLY AS COST v2
# ------------------------------------------------------------

X_external[
    :, :, :N_FEATURES
] = (
    X_external[
        :, :, :N_FEATURES
    ]
    - feat_mean.reshape(
        1, 1, -1
    )
) / feat_std.reshape(
    1, 1, -1
)


# ------------------------------------------------------------
# 10. STATIC NORMALIZATION
# ------------------------------------------------------------

# Very small std in training means feature was constant.
# We still reproduce checkpoint preprocessing exactly.

X_static_external = (
    X_stat_raw
    - stat_mean.reshape(1, -1)
) / stat_std.reshape(1, -1)


# ------------------------------------------------------------
# 11. NUMERICAL AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("PREPROCESSING AUDIT")
print("=" * 90)

print(
    "Temporal shape :",
    X_external.shape
)

print(
    "Static shape   :",
    X_static_external.shape
)

print(
    "Labels shape   :",
    y_external.shape
)

print(
    "\nTemporal NaN:",
    np.isnan(X_external).sum()
)

print(
    "Temporal Inf:",
    np.isinf(X_external).sum()
)

print(
    "Static NaN:",
    np.isnan(X_static_external).sum()
)

print(
    "Static Inf:",
    np.isinf(X_static_external).sum()
)


# ------------------------------------------------------------
# 12. CHANNEL AUDIT
# ------------------------------------------------------------

value_channel = (
    X_external[:, :, :31]
)

mask_channel = (
    X_external[:, :, 31:62]
)

delta_channel = (
    X_external[:, :, 62:93]
)

print("\nChannel ranges:")

print(
    "Value:",
    float(np.nanmin(value_channel)),
    "to",
    float(np.nanmax(value_channel))
)

print(
    "Mask :",
    float(mask_channel.min()),
    "to",
    float(mask_channel.max())
)

print(
    "Delta:",
    float(delta_channel.min()),
    "to",
    float(delta_channel.max())
)

print(
    "\nMask mean:",
    float(mask_channel.mean())
)

print(
    "Delta mean:",
    float(delta_channel.mean())
)


# ------------------------------------------------------------
# 13. IMPORTANT STATIC AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("STATIC NORMALIZATION AUDIT")
print("=" * 90)

for j, f in enumerate(
    STATIC_FEATURES
):

    raw_mean = np.nanmean(
        X_stat_raw[:, j]
    )

    norm_mean = np.nanmean(
        X_static_external[:, j]
    )

    print(
        f"{f:22s} "
        f"raw_mean={raw_mean:9.4f} | "
        f"normalized_mean={norm_mean:12.4f}"
    )


# ------------------------------------------------------------
# 14. FINAL ASSERTIONS
# ------------------------------------------------------------

assert len(X_external) == 12092
assert len(X_static_external) == 12092
assert len(y_external) == 12092

assert not np.isnan(
    X_external
).any()

assert not np.isinf(
    X_external
).any()

if np.isnan(
    X_static_external
).any():

    print(
        "\n⚠ Static array still contains NaN."
    )

    print(
        "Do NOT run model inference yet."
    )

else:

    print(
        "\n✓ Static array contains no NaN"
    )


print("\n" + "=" * 90)
print("✓ eICU COST v2 PREPROCESSING COMPLETE")
print("✓ 93 TEMPORAL INPUT CHANNELS")
print("✓ MIMIC-IV TRAINING NORMALIZATION USED")
print("✓ NO eICU NORMALIZATION FITTING")
print("✓ NO MODEL RETRAINING")
print("=" * 90)

In [ ]:
# ============================================================
# CELL 15 — PRE-INFERENCE FEATURE ALIGNMENT AUDIT
# DO NOT MODIFY DATA
# DO NOT TRAIN
# ============================================================

import numpy as np
import pandas as pd

print("=" * 95)
print("PRE-INFERENCE FEATURE ALIGNMENT AUDIT")
print("=" * 95)

# ============================================================
# A. TEMPORAL FEATURE-BY-FEATURE AUDIT
# ============================================================

print("\nTEMPORAL FEATURES")
print("-" * 95)

rows = []

for j, f in enumerate(TEMPORAL_FEATURES):

    raw = X_temp_raw[:, :, j]
    norm = X_external[:, :, j]

    observed = raw[np.isfinite(raw)]

    if len(observed) > 0:
        raw_min = np.min(observed)
        raw_med = np.median(observed)
        raw_max = np.max(observed)
    else:
        raw_min = raw_med = raw_max = np.nan

    rows.append({
        "idx": j,
        "feature": f,
        "train_mean": float(feat_mean[j]),
        "train_std": float(feat_std[j]),
        "raw_min": raw_min,
        "raw_median": raw_med,
        "raw_max": raw_max,
        "norm_abs_max": float(
            np.max(np.abs(norm))
        )
    })

temp_audit = pd.DataFrame(rows)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

print(temp_audit.to_string(index=False))


# ============================================================
# B. FLAG SUSPICIOUS TEMPORAL FEATURES
# ============================================================

print("\n" + "=" * 95)
print("SUSPICIOUS TEMPORAL FEATURES")
print("=" * 95)

suspicious_temp = temp_audit[
    (temp_audit["train_std"] < 1e-6) |
    (temp_audit["norm_abs_max"] > 100)
].copy()

if len(suspicious_temp):

    print(
        suspicious_temp.to_string(
            index=False
        )
    )

else:
    print("None detected.")


# ============================================================
# C. STATIC FEATURE AUDIT
# ============================================================

print("\n" + "=" * 95)
print("STATIC FEATURES")
print("=" * 95)

rows = []

for j, f in enumerate(STATIC_FEATURES):

    raw = X_stat_raw[:, j]

    finite = raw[np.isfinite(raw)]

    rows.append({
        "idx": j,
        "feature": f,

        "train_mean":
            float(stat_mean[j]),

        "train_std":
            float(stat_std[j]),

        "eicu_missing":
            int(np.isnan(raw).sum()),

        "eicu_min":
            float(np.min(finite))
            if len(finite) else np.nan,

        "eicu_mean":
            float(np.mean(finite))
            if len(finite) else np.nan,

        "eicu_max":
            float(np.max(finite))
            if len(finite) else np.nan,

        "norm_abs_max":
            float(
                np.nanmax(
                    np.abs(
                        X_static_external[:, j]
                    )
                )
            )
    })

static_audit = pd.DataFrame(rows)

print(
    static_audit.to_string(
        index=False
    )
)


# ============================================================
# D. CONSTANT TRAINING FEATURES
# ============================================================

print("\n" + "=" * 95)
print("CONSTANT / NEAR-CONSTANT FEATURES IN MIMIC TRAINING")
print("=" * 95)

constant_temporal = [
    TEMPORAL_FEATURES[j]
    for j in range(31)
    if feat_std[j] < 1e-6
]

constant_static = [
    STATIC_FEATURES[j]
    for j in range(11)
    if stat_std[j] < 1e-6
]

print(
    "Temporal:",
    constant_temporal
)

print(
    "Static  :",
    constant_static
)


# ============================================================
# E. LOCATE STATIC NaNs
# ============================================================

print("\n" + "=" * 95)
print("STATIC NaN LOCATIONS")
print("=" * 95)

nan_pos = np.argwhere(
    np.isnan(X_stat_raw)
)

if len(nan_pos) == 0:

    print("No raw static NaN.")

else:

    for row_idx, feat_idx in nan_pos:

        print(
            "patientunitstayid =",
            ids_final[row_idx],
            "| feature =",
            STATIC_FEATURES[feat_idx],
            "| row =",
            row_idx
        )


# ============================================================
# F. LARGEST NORMALIZED TEMPORAL VALUES
# ============================================================

print("\n" + "=" * 95)
print("TOP 15 TEMPORAL FEATURES BY NORMALIZED MAGNITUDE")
print("=" * 95)

print(
    temp_audit
    .sort_values(
        "norm_abs_max",
        ascending=False
    )
    .head(15)
    [
        [
            "idx",
            "feature",
            "train_mean",
            "train_std",
            "raw_min",
            "raw_median",
            "raw_max",
            "norm_abs_max"
        ]
    ]
    .to_string(index=False)
)


print("\n" + "=" * 95)
print("AUDIT COMPLETE")
print("=" * 95)

print(
    "DO NOT RUN COST INFERENCE YET."
)

print(
    "No values were modified."
)

print(
    "No training was performed."
)

In [ ]:
# ============================================================
# CELL 16 — eICU -> COST TRAINING-SPACE HARMONIZATION
# Frozen external validation
# NO TRAINING / NO FITTING ON eICU
# ============================================================

import numpy as np

print("=" * 90)
print("eICU -> COST TRAINING-SPACE HARMONIZATION")
print("=" * 90)

# Work from RAW arrays, not previously normalized arrays
X_temp_h = X_temp_raw.copy().astype(np.float32)
X_stat_h = X_stat_raw.copy().astype(np.float32)

# ------------------------------------------------------------
# 1. BNP
# Training COST Improved had BNP constant at 0
# ------------------------------------------------------------

bnp_idx = TEMPORAL_FEATURES.index("bnp")

n_bnp_observed = np.isfinite(
    X_temp_h[:, :, bnp_idx]
).sum()

X_temp_h[:, :, bnp_idx] = 0.0

print(
    f"✓ BNP aligned to training state: "
    f"{n_bnp_observed:,} eICU observations -> 0"
)


# ------------------------------------------------------------
# 2. admission_emergency
# Training COST Improved had this feature constant at 0
# ------------------------------------------------------------

adm_em_idx = STATIC_FEATURES.index(
    "admission_emergency"
)

n_emergency_positive = np.nansum(
    X_stat_h[:, adm_em_idx] == 1
)

X_stat_h[:, adm_em_idx] = 0.0

print(
    f"✓ admission_emergency aligned to training state: "
    f"{int(n_emergency_positive):,} positives -> 0"
)


# ------------------------------------------------------------
# 3. Missing gender
# Neutral training-space handling:
# fill raw missing with MIMIC training mean so normalized value = 0
# ------------------------------------------------------------

gender_idx = STATIC_FEATURES.index(
    "gender_male"
)

gender_missing = np.isnan(
    X_stat_h[:, gender_idx]
)

print(
    f"✓ Missing gender values: "
    f"{gender_missing.sum():,}"
)

X_stat_h[
    gender_missing,
    gender_idx
] = stat_mean[gender_idx]


# ------------------------------------------------------------
# 4. FiO2 harmonization
#
# eICU may contain:
#   fraction: 0.21–1.0
#   percent : 21–100
#
# COST/MIMIC FiO2 item is represented in fractional scale
# for model-space compatibility.
#
# Values >1 and <=100 -> divide by 100.
# Implausible values >100 are treated as missing.
# ------------------------------------------------------------

fio2_idx = TEMPORAL_FEATURES.index(
    "fio2"
)

fio2 = X_temp_h[:, :, fio2_idx].copy()

observed = np.isfinite(fio2)

n_fraction = np.sum(
    observed &
    (fio2 > 0) &
    (fio2 <= 1)
)

n_percent = np.sum(
    observed &
    (fio2 > 1) &
    (fio2 <= 100)
)

n_invalid = np.sum(
    observed &
    (
        (fio2 <= 0) |
        (fio2 > 100)
    )
)

# percent -> fraction
pct_mask = (
    np.isfinite(fio2) &
    (fio2 > 1) &
    (fio2 <= 100)
)

fio2[pct_mask] = (
    fio2[pct_mask] / 100.0
)

# invalid -> missing
invalid_mask = (
    np.isfinite(fio2) &
    (
        (fio2 <= 0) |
        (fio2 > 100)
    )
)

fio2[invalid_mask] = np.nan

X_temp_h[:, :, fio2_idx] = fio2

print("\nFiO2:")
print(
    f"  fractional observations : "
    f"{n_fraction:,}"
)
print(
    f"  percent converted       : "
    f"{n_percent:,}"
)
print(
    f"  invalid -> missing      : "
    f"{n_invalid:,}"
)


# ------------------------------------------------------------
# 5. Basic physiological impossible-value cleanup
# Only obvious impossible values; no statistical clipping.
# ------------------------------------------------------------

ranges = {
    "heart_rate":  (1, 300),
    "sbp":         (20, 350),
    "dbp":         (10, 250),
    "map":         (10, 250),
    "resp_rate":   (1, 100),
    "spo2":        (1, 100),
    "temperature": (25, 45),
    "ph":          (6.5, 8.0),
}

print("\nObvious impossible-value cleanup:")

for feature, (lo, hi) in ranges.items():

    j = TEMPORAL_FEATURES.index(feature)

    x = X_temp_h[:, :, j]

    bad = (
        np.isfinite(x) &
        (
            (x < lo) |
            (x > hi)
        )
    )

    n_bad = int(bad.sum())

    X_temp_h[:, :, j][bad] = np.nan

    print(
        f"  {feature:15s}: "
        f"{n_bad:,} -> missing"
    )


# ------------------------------------------------------------
# 6. AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("HARMONIZATION AUDIT")
print("=" * 90)

print(
    "BNP unique:",
    np.unique(
        X_temp_h[:, :, bnp_idx]
    )[:10]
)

print(
    "admission_emergency unique:",
    np.unique(
        X_stat_h[:, adm_em_idx]
    )
)

fio2_valid = (
    X_temp_h[:, :, fio2_idx]
)

fio2_valid = fio2_valid[
    np.isfinite(fio2_valid)
]

if len(fio2_valid):

    print(
        "FiO2 range:",
        float(fio2_valid.min()),
        "to",
        float(fio2_valid.max())
    )

print(
    "Static NaNs:",
    int(np.isnan(X_stat_h).sum())
)

print(
    "Temporal NaNs:",
    f"{np.isnan(X_temp_h).sum():,}"
)

assert np.isnan(X_stat_h).sum() == 0
assert np.all(
    X_temp_h[:, :, bnp_idx] == 0
)
assert np.all(
    X_stat_h[:, adm_em_idx] == 0
)

print("\n✓ TRAINING-SPACE HARMONIZATION COMPLETE")
print("✓ NO eICU STATISTICS FITTED")
print("✓ NO RETRAINING")
print("=" * 90)

In [ ]:
# ============================================================
# CELL 17B — RESUME FINAL PREPROCESSING AFTER LABEL FIX
# ============================================================

# Correct external labels
y_ext = (
    cohort_external[label_col]
    .astype(np.float32)
    .to_numpy()
)

assert len(y_ext) == N

print("=" * 90)
print("FINAL PRE-INFERENCE AUDIT")
print("=" * 90)

print("\nLabels:")
print("  n      :", len(y_ext))
print("  deaths :", int(y_ext.sum()))
print("  rate   :", f"{y_ext.mean()*100:.2f}%")

print("\nShapes:")
print("  Temporal :", X_ext.shape)
print("  Static   :", S_ext.shape)
print("  Labels   :", y_ext.shape)

print("\nNaN / Inf:")
print("  Temporal NaN :", int(np.isnan(X_ext).sum()))
print("  Temporal Inf :", int(np.isinf(X_ext).sum()))
print("  Static NaN   :", int(np.isnan(S_ext).sum()))
print("  Static Inf   :", int(np.isinf(S_ext).sum()))

# ------------------------------------------------------------
# 93-channel audit
# ------------------------------------------------------------

value_ch = X_ext[:, :, :31]
mask_ch  = X_ext[:, :, 31:62]
delta_ch = X_ext[:, :, 62:93]

print("\n93-channel structure:")
print("  Value :", value_ch.shape)
print("  Mask  :", mask_ch.shape)
print("  Delta :", delta_ch.shape)

print("\nRanges:")
print(
    "  Value :",
    float(np.min(value_ch)),
    "to",
    float(np.max(value_ch))
)

print(
    "  Mask  :",
    float(np.min(mask_ch)),
    "to",
    float(np.max(mask_ch))
)

print(
    "  Delta :",
    float(np.min(delta_ch)),
    "to",
    float(np.max(delta_ch))
)

# ------------------------------------------------------------
# Feature-level normalized audit
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("NORMALIZED VALUE FEATURE AUDIT")
print("=" * 90)

for j, feature in enumerate(TEMPORAL_FEATURES):

    x = value_ch[:, :, j]

    print(
        f"{j:2d} {feature:18s} | "
        f"mean={np.mean(x):10.4f} | "
        f"std={np.std(x):10.4f} | "
        f"min={np.min(x):12.4f} | "
        f"max={np.max(x):12.4f}"
    )

print("\n" + "=" * 90)
print("NORMALIZED STATIC FEATURE AUDIT")
print("=" * 90)

for j, feature in enumerate(STATIC_FEATURES):

    x = S_ext[:, j]

    print(
        f"{j:2d} {feature:22s} | "
        f"mean={np.mean(x):10.4f} | "
        f"std={np.std(x):10.4f} | "
        f"min={np.min(x):12.4f} | "
        f"max={np.max(x):12.4f}"
    )

# ------------------------------------------------------------
# Hard assertions
# ------------------------------------------------------------

assert X_ext.shape == (N, 48, 93)
assert S_ext.shape == (N, 11)
assert y_ext.shape == (N,)

assert np.isnan(X_ext).sum() == 0
assert np.isinf(X_ext).sum() == 0

assert np.isnan(S_ext).sum() == 0
assert np.isinf(S_ext).sum() == 0

assert mask_ch.min() >= 0
assert mask_ch.max() <= 1

assert delta_ch.min() >= 0
assert delta_ch.max() <= 1

assert int(y_ext.sum()) == 1489

print("\n" + "=" * 90)
print("✓ FINAL EXTERNAL INPUT ARRAYS READY")
print(f"✓ PATIENTS   = {N:,}")
print("✓ TEMPORAL   = 48 × 93")
print("✓ STATIC     = 11")
print(f"✓ DEATHS     = {int(y_ext.sum()):,}")
print(f"✓ MORTALITY  = {y_ext.mean()*100:.2f}%")
print("✓ NO NaN / Inf")
print("✓ FROZEN MIMIC NORMALIZATION USED")
print("✓ NO eICU NORMALIZATION FITTING")
print("✓ NO RETRAINING")
print("✓ MODEL INFERENCE NOT RUN YET")
print("=" * 90)

In [ ]:
# ============================================================
# FIX y_ext — EXTERNAL VALIDATION LABEL
# ============================================================

print("=" * 90)
print("RECOVERING eICU EXTERNAL VALIDATION LABEL")
print("=" * 90)

# Cari dataframe final cohort yang masih tersedia di runtime
candidate_dfs = []

for name, obj in globals().items():
    if isinstance(obj, pd.DataFrame):
        cols_lower = [str(c).lower() for c in obj.columns]

        if 'patientunitstayid' in cols_lower:
            candidate_dfs.append(
                (name, obj.shape, list(obj.columns))
            )

print("\nCandidate patient-level/data tables:\n")

for name, shape, cols in candidate_dfs:
    print(f"{name:30s} shape={shape}")
    print("  columns:", cols[:30])
    print()

In [ ]:
# ============================================================
# AUDIT 122 PATIENTS WITH MISSING HOSPITAL DISCHARGE STATUS
# DO NOT MODIFY COHORT YET
# ============================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("AUDIT MISSING eICU HOSPITAL MORTALITY OUTCOMES")
print("=" * 90)

# Align exactly with final COST patient order
audit_outcome = (
    cohort_external[
        [
            "patientunitstayid",
            "hospitaldischargestatus",
            "unitdischargestatus",
            "hospitaldischargelocation",
            "unitdischargelocation"
        ]
    ]
    .drop_duplicates("patientunitstayid")
    .set_index("patientunitstayid")
    .reindex(ids_final)
)

missing_hosp = audit_outcome["hospitaldischargestatus"].isna()

print(f"\nFinal COST cohort        : {len(audit_outcome):,}")
print(f"Known hospital outcome   : {(~missing_hosp).sum():,}")
print(f"Missing hospital outcome : {missing_hosp.sum():,}")

print("\n" + "=" * 90)
print("UNIT DISCHARGE STATUS AMONG THE 122 MISSING CASES")
print("=" * 90)

print(
    audit_outcome.loc[
        missing_hosp,
        "unitdischargestatus"
    ].value_counts(dropna=False)
)

print("\n" + "=" * 90)
print("HOSPITAL DISCHARGE LOCATION AMONG MISSING CASES")
print("=" * 90)

print(
    audit_outcome.loc[
        missing_hosp,
        "hospitaldischargelocation"
    ].value_counts(dropna=False)
)

print("\n" + "=" * 90)
print("UNIT DISCHARGE LOCATION AMONG MISSING CASES")
print("=" * 90)

print(
    audit_outcome.loc[
        missing_hosp,
        "unitdischargelocation"
    ].value_counts(dropna=False)
)

# Also verify known outcome distribution
known_status = (
    audit_outcome.loc[
        ~missing_hosp,
        "hospitaldischargestatus"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)

print("\n" + "=" * 90)
print("KNOWN HOSPITAL OUTCOME DISTRIBUTION")
print("=" * 90)

print(known_status.value_counts())

print("\n✓ AUDIT COMPLETE")
print("✓ NO PATIENTS REMOVED YET")
print("✓ NO LABELS IMPUTED")
print("=" * 90)

In [ ]:
# ============================================================
# CELL 17B — FINAL OUTCOME ALIGNMENT FOR EXTERNAL VALIDATION
# Endpoint: eICU in-hospital mortality
#
# Known hospitaldischargestatus only:
#   Alive   = 0
#   Expired = 1
#
# Missing hospital outcome -> excluded from PRIMARY evaluation
# NO outcome imputation
# ============================================================

import numpy as np
import pandas as pd

print("=" * 90)
print("FINAL eICU OUTCOME ALIGNMENT")
print("=" * 90)

# ------------------------------------------------------------
# 1. Align outcomes with exact model-input patient order
# ------------------------------------------------------------

outcome_df = (
    cohort_external[
        ["patientunitstayid", "hospitaldischargestatus"]
    ]
    .drop_duplicates("patientunitstayid")
    .set_index("patientunitstayid")
    .reindex(ids_final)
)

assert len(outcome_df) == len(ids_final)
assert len(outcome_df) == X_ext.shape[0]
assert len(outcome_df) == S_ext.shape[0]

# ------------------------------------------------------------
# 2. Standardize hospital outcome
# ------------------------------------------------------------

status = (
    outcome_df["hospitaldischargestatus"]
    .astype("string")
    .str.strip()
    .str.lower()
)

known_outcome = status.isin(["alive", "expired"]).to_numpy()

print("\nBefore outcome filtering:")
print(f"Patients              : {len(status):,}")
print(f"Known hospital outcome: {known_outcome.sum():,}")
print(f"Missing/unknown       : {(~known_outcome).sum():,}")

# ------------------------------------------------------------
# 3. Keep ONLY known hospital outcomes
# ------------------------------------------------------------

X_ext_eval = X_ext[known_outcome].copy()
S_ext_eval = S_ext[known_outcome].copy()

ids_ext_eval = np.asarray(
    ids_final
)[known_outcome]

status_eval = status.iloc[
    np.where(known_outcome)[0]
]

# ------------------------------------------------------------
# 4. Build binary outcome
# ------------------------------------------------------------

y_ext_eval = (
    status_eval.eq("expired")
    .astype(np.float32)
    .to_numpy()
)

# ------------------------------------------------------------
# 5. Hard alignment checks
# ------------------------------------------------------------

N_eval = len(y_ext_eval)

assert X_ext_eval.shape == (N_eval, 48, 93)
assert S_ext_eval.shape == (N_eval, 11)
assert len(ids_ext_eval) == N_eval

assert np.isnan(X_ext_eval).sum() == 0
assert np.isinf(X_ext_eval).sum() == 0
assert np.isnan(S_ext_eval).sum() == 0
assert np.isinf(S_ext_eval).sum() == 0

assert set(np.unique(y_ext_eval)).issubset({0.0, 1.0})

# ------------------------------------------------------------
# 6. Final audit
# ------------------------------------------------------------

n_death = int(y_ext_eval.sum())
n_alive = int(N_eval - n_death)
mortality = y_ext_eval.mean() * 100

print("\n" + "=" * 90)
print("PRIMARY EXTERNAL VALIDATION COHORT")
print("=" * 90)

print(f"Patients       : {N_eval:,}")
print(f"Alive          : {n_alive:,}")
print(f"Deaths         : {n_death:,}")
print(f"Mortality      : {mortality:.2f}%")

print("\nFinal model inputs:")
print("Temporal :", X_ext_eval.shape)
print("Static   :", S_ext_eval.shape)
print("Labels   :", y_ext_eval.shape)
print("IDs      :", ids_ext_eval.shape)

# Expected from current audit
assert N_eval == 11970
assert n_death == 1489
assert n_alive == 10481

print("\n✓ Hospital mortality endpoint defined")
print("✓ 122 patients with unknown hospital outcome excluded")
print("✓ NO outcome imputation")
print("✓ Patient/input/label alignment preserved")
print("✓ NO RETRAINING")
print("✓ READY FOR FROZEN COST INFERENCE")
print("=" * 90)

In [ ]:
print("=" * 90)
print("MODEL CLASS AUDIT")
print("=" * 90)

# Look for PyTorch model classes / instantiated models
import torch
import torch.nn as nn
import inspect

for name, obj in list(globals().items()):
    try:
        if inspect.isclass(obj) and issubclass(obj, nn.Module):
            print("MODEL CLASS :", name)
        elif isinstance(obj, nn.Module):
            print("MODEL OBJECT:", name, "->", obj.__class__.__name__)
    except:
        pass

In [ ]:
# ============================================================
# CELL 18A — FULL FROZEN COST CHECKPOINT ARCHITECTURE AUDIT
# NO INFERENCE / NO TRAINING
# ============================================================

import torch
from pathlib import Path

CKPT_PATH = Path(
    "/content/drive/MyDrive/COST_PROJECT/mimic-iv-2.1/"
    "saved_models/dsrnn_v2_best.pt"
)

print("=" * 100)
print("COST IMPROVED — FULL CHECKPOINT ARCHITECTURE AUDIT")
print("=" * 100)

assert CKPT_PATH.exists(), f"Checkpoint not found: {CKPT_PATH}"

ckpt = torch.load(
    CKPT_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint:")
print(CKPT_PATH)

print("\nTop-level keys:")
for k in ckpt.keys():
    print(f"  {k}")

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CONFIG")
print("=" * 100)

config = ckpt.get("config", {})

for k, v in config.items():
    print(f"{k:30s}: {v}")

# ------------------------------------------------------------
# STATE DICT
# ------------------------------------------------------------

state = ckpt["model_state_dict"]

print("\n" + "=" * 100)
print("FULL MODEL STATE DICTIONARY")
print("=" * 100)

for i, (name, tensor) in enumerate(state.items()):
    print(
        f"{i:02d}  "
        f"{name:55s} "
        f"{tuple(tensor.shape)}"
    )

# ------------------------------------------------------------
# GROUP PARAMETERS BY TOP-LEVEL MODULE
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("TOP-LEVEL MODULES")
print("=" * 100)

groups = {}

for name in state.keys():
    top = name.split(".")[0]
    groups.setdefault(top, []).append(name)

for module, names in groups.items():

    n_params = sum(
        state[n].numel()
        for n in names
    )

    print(
        f"{module:30s} "
        f"tensors={len(names):3d} | "
        f"parameters={n_params:,}"
    )

# ------------------------------------------------------------
# TOTAL PARAMETERS
# ------------------------------------------------------------

total_params = sum(
    x.numel()
    for x in state.values()
)

print("\nTotal stored parameters/buffers:")
print(f"{total_params:,}")

# ------------------------------------------------------------
# IMPORTANT DIMENSIONS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("KEY ARCHITECTURAL DIMENSIONS")
print("=" * 100)

for key in [
    "input_proj.0.weight",
    "lstm.weight_ih_l0",
    "lstm.weight_hh_l0",
    "lstm.weight_ih_l0_reverse"
]:

    if key in state:
        print(
            f"{key:55s}: "
            f"{tuple(state[key].shape)}"
        )

print("\n" + "=" * 100)
print("✓ CHECKPOINT ARCHITECTURE AUDIT COMPLETE")
print("✓ NO MODEL CREATED")
print("✓ NO INFERENCE")
print("✓ NO TRAINING")
print("=" * 100)

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/COST_PROJECT")

keywords = [
    "dsrnn_v2_best.pt",
    "salience",
    "temporal_attn",
    "bidir_proj",
]

print("=" * 90)
print("SEARCHING ORIGINAL COST MODEL CODE")
print("=" * 90)

hits = []

for p in list(ROOT.rglob("*.py")) + list(ROOT.rglob("*.ipynb")):
    try:
        txt = p.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        found = [k for k in keywords if k in txt]

        if found:
            hits.append((p, found))

    except Exception:
        pass

for p, found in hits:
    print("\nFILE:", p)
    print("FOUND:", found)

print("\nTotal candidate files:", len(hits))

In [ ]:
# ============================================================
# CELL 18 — LOAD EXACT COST IMPROVED ARCHITECTURE
# + STRICT FROZEN CHECKPOINT LOADING
# NO TRAINING
# NO INFERENCE YET
# ============================================================

import math
import torch
import torch.nn as nn
from pathlib import Path

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

CKPT_PATH = Path(
    "/content/drive/MyDrive/COST_PROJECT/mimic-iv-2.1/"
    "saved_models/dsrnn_v2_best.pt"
)

print("=" * 100)
print("LOADING EXACT COST IMPROVED MODEL")
print("=" * 100)

print("Device     :", DEVICE)
print("Checkpoint :", CKPT_PATH)

assert CKPT_PATH.exists(), (
    f"Checkpoint not found: {CKPT_PATH}"
)

# ============================================================
# 1. LOAD CHECKPOINT
# ============================================================

ckpt = torch.load(
    CKPT_PATH,
    map_location="cpu",
    weights_only=False
)

cfg = ckpt.get("config", {})


# ============================================================
# 2. EXACT ORIGINAL MODEL COMPONENTS
# ============================================================

class SinusoidalPE(nn.Module):

    def __init__(
        self,
        d_model,
        max_len=256,
        dropout=0.1
    ):
        super().__init__()

        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(
            max_len,
            d_model
        )

        pos = (
            torch.arange(max_len)
            .unsqueeze(1)
            .float()
        )

        div = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            ).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(
            pos * div
        )

        pe[:, 1::2] = torch.cos(
            pos * div
        )

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )

    def forward(self, x):

        return self.dropout(
            x + self.pe[:, :x.size(1)]
        )


class StochasticDepth(nn.Module):

    def __init__(self, p=0.1):

        super().__init__()

        self.p = p

    def forward(
        self,
        x,
        residual
    ):

        if (
            not self.training
            or self.p == 0
        ):
            return x + residual

        keep = torch.bernoulli(
            torch.tensor(
                1 - self.p
            )
        ).item()

        return x + residual * keep


class ClinicalSalienceV2(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        n_features
    ):

        super().__init__()

        self.fc1 = nn.Linear(
            input_dim,
            hidden_dim * 2
        )

        self.norm = nn.LayerNorm(
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            n_features
        )

        self.tau = nn.Parameter(
            torch.tensor(0.5)
        )

    def forward(
        self,
        x_t,
        h_prev
    ):

        h = self.fc1(
            torch.cat(
                [x_t, h_prev],
                -1
            )
        )

        h, gate = h.chunk(
            2,
            -1
        )

        h = self.norm(
            h * torch.sigmoid(gate)
        )

        s = torch.sigmoid(
            self.fc2(h)
        )

        hard = (
            s > self.tau
        ).float()

        mask = (
            hard
            - s.detach()
            + s
        )

        return mask, s


class TemporalMultiHeadAttn(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        dropout=0.1
    ):

        super().__init__()

        self.attn = nn.MultiheadAttention(
            d_model,
            n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm = nn.LayerNorm(
            d_model
        )

        self.sdrop = StochasticDepth(
            0.1
        )

    def forward(self, x):

        out, w = self.attn(
            x,
            x,
            x
        )

        out = self.norm(
            self.sdrop(
                x,
                out
            )
        )

        return out, w


class StaticTemporalFusion(nn.Module):

    def __init__(
        self,
        d_temporal,
        n_static,
        d_out,
        n_heads=4,
        dropout=0.1
    ):

        super().__init__()

        self.static_proj = nn.Linear(
            n_static,
            d_temporal
        )

        self.cross_attn = (
            nn.MultiheadAttention(
                d_temporal,
                n_heads,
                dropout=dropout,
                batch_first=True
            )
        )

        self.norm = nn.LayerNorm(
            d_temporal
        )

        self.ffn = nn.Sequential(

            nn.Linear(
                d_temporal * 2,
                d_temporal
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                d_temporal,
                d_out
            )
        )

    def forward(
        self,
        s,
        h
    ):

        q = self.static_proj(
            s
        ).unsqueeze(1)

        ctx, w = self.cross_attn(
            q,
            h,
            h
        )

        ctx = self.norm(
            ctx.squeeze(1)
        )

        z = self.ffn(
            torch.cat(
                [
                    ctx,
                    h.mean(1)
                ],
                -1
            )
        )

        return z, w.squeeze(1)


# ============================================================
# 3. EXACT COST IMPROVED / DSRNNv2
# ============================================================

class DSRNNv2(nn.Module):

    def __init__(
        self,
        n_input=93,
        n_static=11,
        hidden_size=256,
        n_lstm_layers=2,
        n_heads=4,
        attention_dim=128,
        salience_hidden=128,
        dropout=0.3,
        lambda_sparse=5e-4
    ):

        super().__init__()

        self.hidden_size = hidden_size
        self.lambda_sparse = lambda_sparse

        self.input_proj = nn.Sequential(

            nn.Linear(
                n_input,
                hidden_size
            ),

            nn.LayerNorm(
                hidden_size
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            )
        )

        self.pos_enc = SinusoidalPE(
            hidden_size,
            dropout=dropout * 0.5
        )

        self.salience = ClinicalSalienceV2(
            hidden_size * 2,
            salience_hidden,
            hidden_size
        )

        self.lstm = nn.LSTM(
            hidden_size,
            hidden_size,
            num_layers=n_lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        self.bidir_proj = nn.Sequential(

            nn.Linear(
                hidden_size * 2,
                hidden_size
            ),

            nn.LayerNorm(
                hidden_size
            ),

            nn.GELU()
        )

        self.temporal_attn = (
            TemporalMultiHeadAttn(
                hidden_size,
                n_heads,
                dropout
            )
        )

        self.fusion = StaticTemporalFusion(
            hidden_size,
            n_static,
            attention_dim,
            n_heads,
            dropout
        )

        self.classifier = nn.Sequential(

            nn.LayerNorm(
                attention_dim
            ),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                attention_dim,
                attention_dim // 2
            ),

            nn.GELU(),

            nn.Dropout(
                dropout * 0.5
            ),

            nn.Linear(
                attention_dim // 2,
                1
            )
        )

    def forward(
        self,
        x_temporal,
        x_static
    ):

        b, seq, _ = (
            x_temporal.shape
        )

        xp = self.pos_enc(
            self.input_proj(
                x_temporal
            )
        )

        hp = torch.zeros(
            b,
            self.hidden_size,
            device=x_temporal.device
        )

        gs = []

        sp = torch.tensor(
            0.0,
            device=x_temporal.device
        )

        for t in range(seq):

            xt = xp[:, t]

            m, _ = self.salience(
                xt,
                hp
            )

            xg = (
                m * xt
                + (1 - m) * xt * 0.1
            )

            gs.append(
                xg.unsqueeze(1)
            )

            hp = (
                hp * 0.9
                + xg.detach() * 0.1
            )

            sp += m.norm(
                p=1
            )

        g = torch.cat(
            gs,
            1
        )

        sp = (
            self.lambda_sparse
            * sp
            / (seq * b)
        )

        lo, _ = self.lstm(
            g
        )

        lo = self.bidir_proj(
            lo
        )

        H, _ = self.temporal_attn(
            lo
        )

        z, _ = self.fusion(
            x_static,
            H
        )

        logits = self.classifier(
            z
        ).squeeze(-1)

        return logits, sp


# ============================================================
# 4. INSTANTIATE FROM ORIGINAL CHECKPOINT CONFIG
# ============================================================

model = DSRNNv2(

    n_input=93,
    n_static=11,

    hidden_size=cfg.get(
        "hidden_size",
        256
    ),

    n_lstm_layers=cfg.get(
        "n_lstm_layers",
        2
    ),

    n_heads=cfg.get(
        "n_heads",
        4
    ),

    attention_dim=cfg.get(
        "attention_dim",
        128
    ),

    salience_hidden=cfg.get(
        "salience_hidden",
        128
    ),

    dropout=cfg.get(
        "dropout",
        0.3
    ),

    lambda_sparse=cfg.get(
        "lambda_sparse",
        5e-4
    )

).to(DEVICE)


# ============================================================
# 5. STRICT CHECKPOINT LOAD
# ============================================================

state = ckpt[
    "model_state_dict"
]

load_result = model.load_state_dict(
    state,
    strict=True
)

model.eval()


# ============================================================
# 6. AUDIT
# ============================================================

n_params = sum(
    p.numel()
    for p in model.parameters()
)

n_trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\n" + "=" * 100)
print("STRICT CHECKPOINT LOAD RESULT")
print("=" * 100)

print("Result:", load_result)

print(
    f"Model parameters : "
    f"{n_params:,}"
)

print(
    f"Trainable params : "
    f"{n_trainable:,}"
)

print(
    "Model mode       :",
    "EVAL"
    if not model.training
    else "TRAIN"
)

print(
    "Input temporal   : 48 × 93"
)

print(
    "Input static     : 11"
)

print("\n✓ EXACT DSRNNv2 ARCHITECTURE CREATED")
print("✓ CHECKPOINT LOADED WITH strict=True")
print("✓ model.eval() ENABLED")
print("✓ NO TRAINING PERFORMED")
print("✓ NO EXTERNAL INFERENCE PERFORMED YET")
print("=" * 100)

In [ ]:
# CELL 18B — FINAL READINESS CHECK

print("=" * 80)
print("FINAL READINESS CHECK BEFORE eICU INFERENCE")
print("=" * 80)

print("X_ext_eval :", X_ext_eval.shape)
print("S_ext_eval :", S_ext_eval.shape)
print("y_ext_eval :", y_ext_eval.shape)

print("Patients   :", len(y_ext_eval))
print("Deaths     :", int(y_ext_eval.sum()))
print("Mortality  :", f"{y_ext_eval.mean()*100:.2f}%")

print("Model eval :", not model.training)
print("Device     :", DEVICE)

assert X_ext_eval.shape == (11970, 48, 93)
assert S_ext_eval.shape == (11970, 11)
assert y_ext_eval.shape == (11970,)
assert not model.training

print("\n✓ READY FOR EXTERNAL INFERENCE")

In [ ]:
# ============================================================
# CELL 19 — FROZEN COST IMPROVED EXTERNAL VALIDATION
# eICU — FINAL INFERENCE
# NO RETRAINING
# ============================================================

import numpy as np
import torch

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    brier_score_loss
)

print("=" * 100)
print("COST IMPROVED — eICU EXTERNAL VALIDATION")
print("=" * 100)

# ============================================================
# 1. FINAL INPUT CHECK
# ============================================================

X = X_ext_eval.astype(np.float32)
S = S_ext_eval.astype(np.float32)
y = y_ext_eval.astype(int)

assert X.shape == (11970, 48, 93)
assert S.shape == (11970, 11)
assert y.shape == (11970,)

assert np.isfinite(X).all()
assert np.isfinite(S).all()

print(f"Patients  : {len(y):,}")
print(f"Deaths    : {int(y.sum()):,}")
print(f"Mortality : {y.mean()*100:.2f}%")
print(f"Device    : {DEVICE}")

# ============================================================
# 2. FROZEN MODEL INFERENCE
# ============================================================

model.eval()

BATCH_SIZE = 256
prob_list = []

print("\nRunning frozen COST Improved inference...")

with torch.no_grad():

    for start in range(0, len(y), BATCH_SIZE):

        end = min(start + BATCH_SIZE, len(y))

        xb = torch.from_numpy(
            X[start:end]
        ).to(DEVICE)

        sb = torch.from_numpy(
            S[start:end]
        ).to(DEVICE)

        logits, _ = model(xb, sb)

        probs = torch.sigmoid(logits)

        prob_list.append(
            probs.detach().cpu().numpy()
        )

        if end % 2048 < BATCH_SIZE or end == len(y):
            print(
                f"Processed: {end:,}/{len(y):,}"
            )

y_prob_ext = np.concatenate(
    prob_list
).reshape(-1)

y_true_ext = y.copy()

assert len(y_prob_ext) == len(y_true_ext)
assert np.isfinite(y_prob_ext).all()

# ============================================================
# 3. PREDICTION AUDIT
# ============================================================

print("\n" + "=" * 100)
print("PREDICTION AUDIT")
print("=" * 100)

print(
    f"Probability min    : {y_prob_ext.min():.6f}"
)

print(
    f"Probability median : {np.median(y_prob_ext):.6f}"
)

print(
    f"Probability mean   : {y_prob_ext.mean():.6f}"
)

print(
    f"Probability max    : {y_prob_ext.max():.6f}"
)

# ============================================================
# 4. PRIMARY THRESHOLD-INDEPENDENT METRICS
# ============================================================

auroc = roc_auc_score(
    y_true_ext,
    y_prob_ext
)

auprc = average_precision_score(
    y_true_ext,
    y_prob_ext
)

brier = brier_score_loss(
    y_true_ext,
    y_prob_ext
)

# ============================================================
# 5. SECONDARY THRESHOLD ANALYSIS — YOUDEN
# ============================================================

fpr, tpr, thresholds = roc_curve(
    y_true_ext,
    y_prob_ext
)

youden_index = np.argmax(
    tpr - fpr
)

youden_threshold = thresholds[
    youden_index
]

y_pred_ext = (
    y_prob_ext >= youden_threshold
).astype(int)

# ============================================================
# 6. CONFUSION MATRIX + METRICS
# ============================================================

tn, fp, fn, tp = confusion_matrix(
    y_true_ext,
    y_pred_ext
).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

ppv = (
    tp / (tp + fp)
    if (tp + fp) > 0
    else np.nan
)

npv = (
    tn / (tn + fn)
    if (tn + fn) > 0
    else np.nan
)

accuracy = accuracy_score(
    y_true_ext,
    y_pred_ext
)

precision = precision_score(
    y_true_ext,
    y_pred_ext,
    zero_division=0
)

recall = recall_score(
    y_true_ext,
    y_pred_ext,
    zero_division=0
)

f1 = f1_score(
    y_true_ext,
    y_pred_ext,
    zero_division=0
)

mcc = matthews_corrcoef(
    y_true_ext,
    y_pred_ext
)

# ============================================================
# 7. DISPLAY EXTERNAL VALIDATION RESULTS
# ============================================================

print("\n" + "=" * 100)
print("PRIMARY eICU EXTERNAL VALIDATION")
print("=" * 100)

print(f"AUROC             : {auroc:.4f}")
print(f"AUPRC             : {auprc:.4f}")
print(f"Brier score       : {brier:.4f}")

print("\nSecondary threshold-based metrics")
print("-" * 100)

print(
    f"Youden threshold  : {youden_threshold:.6f}"
)

print(f"Accuracy          : {accuracy:.4f}")
print(f"Sensitivity       : {sensitivity:.4f}")
print(f"Specificity       : {specificity:.4f}")
print(f"PPV / Precision   : {ppv:.4f}")
print(f"NPV               : {npv:.4f}")
print(f"F1-score          : {f1:.4f}")
print(f"MCC               : {mcc:.4f}")

print("\nConfusion matrix")
print("-" * 100)

print(f"TN : {tn:,}")
print(f"FP : {fp:,}")
print(f"FN : {fn:,}")
print(f"TP : {tp:,}")

# ============================================================
# 8. INTERNAL vs EXTERNAL AUROC
# ============================================================

internal_auc = float(
    ckpt.get(
        "test_auc_ensemble",
        np.nan
    )
)

print("\n" + "=" * 100)
print("INTERNAL vs EXTERNAL")
print("=" * 100)

print(
    f"MIMIC-IV internal test AUROC : "
    f"{internal_auc:.4f}"
)

print(
    f"eICU external AUROC          : "
    f"{auroc:.4f}"
)

if np.isfinite(internal_auc):

    print(
        f"AUROC difference             : "
        f"{auroc - internal_auc:+.4f}"
    )

# ============================================================
# 9. STORE RESULTS
# ============================================================

external_results = {
    "n": len(y_true_ext),
    "deaths": int(y_true_ext.sum()),
    "mortality": float(y_true_ext.mean()),
    "auroc": float(auroc),
    "auprc": float(auprc),
    "brier": float(brier),
    "youden_threshold": float(youden_threshold),
    "accuracy": float(accuracy),
    "sensitivity": float(sensitivity),
    "specificity": float(specificity),
    "ppv": float(ppv),
    "npv": float(npv),
    "f1": float(f1),
    "mcc": float(mcc),
    "tn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "tp": int(tp)
}

print("\n" + "=" * 100)
print("✓ FROZEN COST IMPROVED INFERENCE COMPLETE")
print("✓ eICU EXTERNAL VALIDATION COMPLETE")
print("✓ NO RETRAINING")
print("✓ NO FINE-TUNING")
print("✓ NO eICU NORMALIZATION FITTING")
print("=" * 100)

In [ ]:
# ============================================================
# CELL 21 — SEARCH FROZEN MIMIC MICE / IMPUTER ARTIFACT
# READ ONLY
# NO TRAINING
# NO INFERENCE
# ============================================================

from pathlib import Path
import torch
import pickle
import joblib
import os

ROOT = Path("/content/drive/MyDrive/COST_PROJECT")

print("=" * 100)
print("SEARCHING FOR SAVED MIMIC TRAINING IMPUTER / MICE ARTIFACT")
print("=" * 100)

# ------------------------------------------------------------
# 1. Search filenames that may contain saved imputer
# ------------------------------------------------------------

keywords = [
    "imputer",
    "mice",
    "iterative",
    "preprocess",
    "norm_stats",
    "preprocessing",
]

extensions = [
    ".pkl", ".pickle", ".joblib",
    ".pt", ".pth", ".npz", ".npy"
]

hits = []

for p in ROOT.rglob("*"):

    if not p.is_file():
        continue

    name = p.name.lower()

    if (
        any(k in name for k in keywords)
        and p.suffix.lower() in extensions
    ):
        hits.append(p)

print("\nPotential preprocessing artifacts:")
if hits:
    for p in hits:
        try:
            size_mb = p.stat().st_size / (1024**2)
        except:
            size_mb = -1

        print(f"  {p} | {size_mb:.2f} MB")
else:
    print("  NONE FOUND BY FILENAME")


# ------------------------------------------------------------
# 2. Inspect COST checkpoint
# ------------------------------------------------------------

CKPT_PATH = (
    "/content/drive/MyDrive/COST_PROJECT/"
    "mimic-iv-2.1/saved_models/dsrnn_v2_best.pt"
)

print("\n" + "=" * 100)
print("CHECKPOINT CONTENT")
print("=" * 100)

ckpt = torch.load(
    CKPT_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nTop-level checkpoint keys:")

for k in ckpt.keys():

    obj = ckpt[k]

    print(
        f"  {k:30s} "
        f"type={type(obj).__name__}"
    )

    if isinstance(obj, dict):
        print(
            "     subkeys:",
            list(obj.keys())[:30]
        )


# ------------------------------------------------------------
# 3. Specifically search checkpoint recursively
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("SEARCHING CHECKPOINT FOR IMPUTER-LIKE OBJECTS")
print("=" * 100)

found_objects = []

def inspect_object(obj, path="root"):

    typename = type(obj).__name__.lower()

    if any(
        x in typename
        for x in [
            "imputer",
            "iterative",
            "scaler"
        ]
    ):
        found_objects.append(
            (path, type(obj).__name__)
        )

    if isinstance(obj, dict):

        for key, value in obj.items():

            key_lower = str(key).lower()

            if any(
                x in key_lower
                for x in [
                    "imputer",
                    "mice",
                    "iterative"
                ]
            ):
                found_objects.append(
                    (
                        f"{path}.{key}",
                        type(value).__name__
                    )
                )

            inspect_object(
                value,
                f"{path}.{key}"
            )

inspect_object(ckpt)

if found_objects:

    for path, typ in found_objects:
        print(
            f"  FOUND: {path} | type={typ}"
        )

else:
    print(
        "  No saved imputer-like object "
        "found inside checkpoint."
    )


# ------------------------------------------------------------
# 4. Check whether original training arrays still exist
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("SEARCHING FOR ORIGINAL MIMIC TRAINING ARRAYS")
print("=" * 100)

array_keywords = [
    "temporal",
    "x_temp",
    "x_temporal",
    "train",
    "mimic"
]

array_hits = []

for p in ROOT.rglob("*"):

    if not p.is_file():
        continue

    name = p.name.lower()

    if (
        p.suffix.lower()
        in [".npy", ".npz", ".pkl", ".pickle", ".joblib"]
        and any(k in name for k in array_keywords)
    ):

        array_hits.append(p)

# avoid huge output
for p in array_hits[:100]:

    try:
        size_mb = p.stat().st_size / (1024**2)
    except:
        size_mb = -1

    print(
        f"  {p} | {size_mb:.2f} MB"
    )

print(
    f"\nTotal candidate array files: "
    f"{len(array_hits)}"
)


print("\n" + "=" * 100)
print("SEARCH COMPLETE")
print("=" * 100)

print("""
DO NOT RUN EXTERNAL INFERENCE AGAIN YET.

We need to determine:

1. Was the fitted MIMIC IterativeImputer saved?
2. If not, are the original MIMIC training arrays available?
3. Can the exact training imputer be reconstructed WITHOUT using eICU outcomes?
""")

In [ ]:
# ============================================================
# NEXT CELL — eICU EXTERNAL VALIDATION: FINAL STATISTICAL AUDIT
# Bootstrap 95% CI + calibration + publication-ready summary
# NO RETRAINING / NO RE-INFERENCE
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)
from sklearn.linear_model import LogisticRegression

print("=" * 100)
print("COST IMPROVED — eICU EXTERNAL VALIDATION STATISTICAL AUDIT")
print("=" * 100)


# ============================================================
# 1. LOCATE EXISTING PREDICTION ARRAY
# ============================================================

# y_ext_eval already exists from the completed external validation.
y_true = np.asarray(y_ext_eval).astype(int).reshape(-1)

# Find the probability variable produced by the previous inference cell.
candidate_names = [
    "y_prob_ext",
    "y_prob",
    "probs_ext",
    "probs",
    "pred_probs",
    "y_pred_prob",
    "external_probs",
    "prob_ext",
]

y_prob = None
used_name = None

for name in candidate_names:
    if name in globals():
        arr = np.asarray(globals()[name]).reshape(-1)

        if (
            len(arr) == len(y_true)
            and np.all(np.isfinite(arr))
            and np.min(arr) >= 0
            and np.max(arr) <= 1
        ):
            y_prob = arr.astype(np.float64)
            used_name = name
            break

if y_prob is None:
    raise RuntimeError(
        "Probability array from the previous inference cell was not found.\n"
        "Run the external inference cell immediately before this cell, "
        "or tell me the probability variable name."
    )

print(f"\nPrediction variable : {used_name}")
print(f"Patients            : {len(y_true):,}")
print(f"Deaths              : {int(y_true.sum()):,}")
print(f"Mortality           : {y_true.mean()*100:.2f}%")

assert len(y_true) == len(y_prob)
assert set(np.unique(y_true)).issubset({0, 1})


# ============================================================
# 2. POINT ESTIMATES
# ============================================================

auc = roc_auc_score(y_true, y_prob)
auprc = average_precision_score(y_true, y_prob)
brier = brier_score_loss(y_true, y_prob)

print("\n" + "=" * 100)
print("POINT ESTIMATES")
print("=" * 100)

print(f"AUROC       : {auc:.4f}")
print(f"AUPRC       : {auprc:.4f}")
print(f"Brier score : {brier:.4f}")


# ============================================================
# 3. PATIENT-LEVEL BOOTSTRAP 95% CI
# ============================================================

N_BOOT = 2000
SEED = 42

rng = np.random.default_rng(SEED)

boot_auc = []
boot_auprc = []
boot_brier = []

n = len(y_true)

print("\nRunning 2,000 bootstrap iterations...")

for b in range(N_BOOT):

    idx = rng.integers(0, n, n)

    y_b = y_true[idx]
    p_b = y_prob[idx]

    # ROC AUC requires both outcome classes
    if len(np.unique(y_b)) < 2:
        continue

    boot_auc.append(
        roc_auc_score(y_b, p_b)
    )

    boot_auprc.append(
        average_precision_score(y_b, p_b)
    )

    boot_brier.append(
        brier_score_loss(y_b, p_b)
    )

    if (b + 1) % 500 == 0:
        print(f"  completed {b+1:,}/{N_BOOT:,}")


def percentile_ci(x):
    x = np.asarray(x)
    return (
        np.percentile(x, 2.5),
        np.percentile(x, 97.5)
    )


auc_ci = percentile_ci(boot_auc)
auprc_ci = percentile_ci(boot_auprc)
brier_ci = percentile_ci(boot_brier)


print("\n" + "=" * 100)
print("BOOTSTRAP 95% CONFIDENCE INTERVALS")
print("=" * 100)

print(
    f"AUROC       : {auc:.4f} "
    f"(95% CI {auc_ci[0]:.4f}–{auc_ci[1]:.4f})"
)

print(
    f"AUPRC       : {auprc:.4f} "
    f"(95% CI {auprc_ci[0]:.4f}–{auprc_ci[1]:.4f})"
)

print(
    f"Brier score : {brier:.4f} "
    f"(95% CI {brier_ci[0]:.4f}–{brier_ci[1]:.4f})"
)


# ============================================================
# 4. CALIBRATION INTERCEPT AND SLOPE
# ============================================================

eps = 1e-7

p_clip = np.clip(
    y_prob,
    eps,
    1 - eps
)

logit_p = np.log(
    p_clip / (1 - p_clip)
).reshape(-1, 1)


# Calibration intercept:
# logistic model with fixed slope approximately represented
# through intercept-only correction of the prediction logit.

# Calibration slope + intercept
cal_model = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000
)

cal_model.fit(logit_p, y_true)

cal_intercept = float(cal_model.intercept_[0])
cal_slope = float(cal_model.coef_[0, 0])


print("\n" + "=" * 100)
print("CALIBRATION")
print("=" * 100)

print(f"Calibration intercept : {cal_intercept:.4f}")
print(f"Calibration slope     : {cal_slope:.4f}")


# ============================================================
# 5. EXPECTED CALIBRATION ERROR (ECE)
# ============================================================

def expected_calibration_error(y, p, n_bins=10):

    bins = np.linspace(0.0, 1.0, n_bins + 1)

    ece = 0.0

    for i in range(n_bins):

        if i == n_bins - 1:
            mask = (
                (p >= bins[i]) &
                (p <= bins[i + 1])
            )
        else:
            mask = (
                (p >= bins[i]) &
                (p < bins[i + 1])
            )

        if mask.sum() == 0:
            continue

        observed = y[mask].mean()
        predicted = p[mask].mean()

        ece += (
            mask.mean()
            * abs(observed - predicted)
        )

    return ece


ece = expected_calibration_error(
    y_true,
    y_prob,
    n_bins=10
)

print(f"ECE (10 bins)         : {ece:.4f}")


# ============================================================
# 6. INTERNAL vs EXTERNAL SUMMARY
# ============================================================

# Frozen checkpoint/internal result already reported:
internal_auc = 0.8025

auc_drop = auc - internal_auc


print("\n" + "=" * 100)
print("INTERNAL vs EXTERNAL GENERALIZATION")
print("=" * 100)

print(f"MIMIC-IV internal AUROC : {internal_auc:.4f}")
print(f"eICU external AUROC     : {auc:.4f}")
print(f"Absolute difference     : {auc_drop:+.4f}")


# ============================================================
# 7. PUBLICATION-READY TABLE
# ============================================================

summary = pd.DataFrame({

    "Cohort": [
        "MIMIC-IV Internal Test",
        "eICU External Validation"
    ],

    "N": [
        1044,
        len(y_true)
    ],

    "Deaths": [
        162,
        int(y_true.sum())
    ],

    "Mortality (%)": [
        15.5,
        y_true.mean() * 100
    ],

    "AUROC": [
        internal_auc,
        auc
    ],

    "AUROC 95% CI": [
        "Internal repeated-run result",
        f"{auc_ci[0]:.4f}–{auc_ci[1]:.4f}"
    ],

    "AUPRC": [
        np.nan,
        auprc
    ],

    "Brier": [
        np.nan,
        brier
    ]
})


print("\n" + "=" * 100)
print("PUBLICATION-READY EXTERNAL VALIDATION SUMMARY")
print("=" * 100)

print(summary.to_string(index=False))


# ============================================================
# 8. SAVE RESULTS
# ============================================================

external_stats = {
    "n": int(len(y_true)),
    "deaths": int(y_true.sum()),
    "mortality": float(y_true.mean()),

    "auroc": float(auc),
    "auroc_ci_low": float(auc_ci[0]),
    "auroc_ci_high": float(auc_ci[1]),

    "auprc": float(auprc),
    "auprc_ci_low": float(auprc_ci[0]),
    "auprc_ci_high": float(auprc_ci[1]),

    "brier": float(brier),
    "brier_ci_low": float(brier_ci[0]),
    "brier_ci_high": float(brier_ci[1]),

    "calibration_intercept": cal_intercept,
    "calibration_slope": cal_slope,
    "ece_10bin": float(ece),

    "internal_auc": internal_auc,
    "external_internal_auc_difference": float(auc_drop)
}


print("\n" + "=" * 100)
print("✓ EXTERNAL VALIDATION STATISTICAL AUDIT COMPLETE")
print("✓ 95% CI COMPUTED")
print("✓ CALIBRATION QUANTIFIED")
print("✓ INTERNAL–EXTERNAL GENERALIZATION QUANTIFIED")
print("✓ NO RETRAINING")
print("✓ NO FINE-TUNING")
print("✓ NO NEW INFERENCE")
print("=" * 100)